# Global atlas analysis

## Repository navigation

This notebook is a downstream analytical workflow record for the integrated cross-disease immune-cell atlas. It starts from a processed, integrated AnnData object and records the analysis logic used for metadata harmonization, QC and integration-quality inspection, atlas visualization, cytokine-storm scoring, statistical summaries, CellChat input preparation, and SLE pDC analyses.

# 1. Load processed integrated atlas object

In [1]:
import os

# Set environment variables to limit the number of threads used by various libraries
os.environ["OMP_NUM_THREADS"] = "16"
os.environ["OPENBLAS_NUM_THREADS"] = "16"
os.environ["MKL_NUM_THREADS"] = "16"
os.environ["VECLIB_MAXIMUM_THREADS"] = "16"
os.environ["NUMEXPR_NUM_THREADS"] = "16"

# Documented private input and output path variables
# The processed AnnData object and sample metadata workbook are not distributed
# ATLAS_H5AD, SAMPLE_METADATA_XLSX, and ATLAS_OUTPUT_DIR record the expected
# environment-variable names used in the original downstream workflow

analysis_start_dir = globals().get("analysis_start_dir", os.getcwd())

def resolve_analysis_path(path_setting):
    return path_setting if os.path.isabs(path_setting) else os.path.join(analysis_start_dir, path_setting)

atlas_h5ad_path = resolve_analysis_path(
    os.environ.get("ATLAS_H5AD", "adata_integrated.h5ad")
)
sample_metadata_path = resolve_analysis_path(
    os.environ.get("SAMPLE_METADATA_XLSX", "Sample_infor_clear.xlsx")
)
atlas_output_dir = resolve_analysis_path(
    os.environ.get("ATLAS_OUTPUT_DIR", os.path.join("outputs", "global_atlas"))
)
cellchat_root = resolve_analysis_path(
    os.environ.get("CELLCHAT_ROOT", os.path.join("outputs", "cellchat"))
)
cellchat_car_t_input_dir = resolve_analysis_path(
    os.environ.get("CELLCHAT_CAR_T_INPUT_DIR", os.path.join(cellchat_root, "cart", "input"))
)
cellchat_covid19_input_dir = resolve_analysis_path(
    os.environ.get("CELLCHAT_COVID19_INPUT_DIR", os.path.join(cellchat_root, "covid19", "input"))
)
cellchat_sle_input_dir = resolve_analysis_path(
    os.environ.get("CELLCHAT_SLE_INPUT_DIR", os.path.join(cellchat_root, "sle", "input"))
)

os.makedirs(atlas_output_dir, exist_ok=True)
RANDOM_SEED = 42

In [3]:
# Dependencies and requirements
# Python kernel used during development: Python 3.11.13

import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
import matplotlib.pyplot as plt
from matplotlib import cm, colors
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import plotly.graph_objects as go
import scipy.io
from scipy import sparse
from scipy.sparse import csc_matrix, csr_matrix
from scipy.stats import pearsonr, spearmanr
from scipy.stats import entropy
from pathlib import Path
import gseapy as gp
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font

In [4]:
# Global figure settings
sc.settings.set_figure_params(frameon=False, dpi=300)
sc.set_figure_params(dpi_save=300)
sc.settings.verbosity = 2

In [ ]:
# Load the integrated, annotated AnnData object and direct relative outputs to
# the configured global-atlas output directory
adata_main = sc.read_h5ad(atlas_h5ad_path)

In [ ]:
required_obs_columns = [
    'sample_id', 'disease', 'stage', 'severity', 
    'leiden1_anno', 'subcluster_anno',
    'n_genes_by_counts', 'total_counts', 'pct_counts_mt'
]
missing_obs_columns = [col for col in required_obs_columns if col not in adata_main.obs.columns]
if missing_obs_columns:
    raise ValueError(f"Missing required atlas metadata columns: {missing_obs_columns}")
if not adata_main.obs_names.is_unique:
    raise ValueError('adata_main.obs_names must be unique.')
if not adata_main.var_names.is_unique:
    raise ValueError('adata_main.var_names must be unique.')
for embedding in ['X_umap', 'X_pca']:
    if embedding not in adata_main.obsm:
        raise ValueError(f"Missing required embedding: {embedding}")

expected_diseases = {'CAR-T_CRS', 'COVID19', 'SLE', 'HC'}
missing_diseases = expected_diseases - set(adata_main.obs['disease'].dropna().astype(str))
if missing_diseases:
    raise ValueError(f"Missing expected disease categories: {sorted(missing_diseases)}")
expected_stages = {
    'CAR-T_CRS_before', 'CAR-T_CRS_pro', 'CAR-T_CRS_con',
    'COVID19_pro', 'COVID19_con', 'SLE', 'HC'
}
missing_stages = expected_stages - set(adata_main.obs['stage'].dropna().astype(str))
if missing_stages:
    raise ValueError(f"Missing expected stage categories: {sorted(missing_stages)}")
expected_severities = {'Moderate', 'Severe', 'No_CRS', 'HC'}
missing_severities = expected_severities - set(
    adata_main.obs['severity'].dropna().astype(str)
)
if missing_severities:
    raise ValueError(f"Missing expected severity categories: {sorted(missing_severities)}")

os.chdir(atlas_output_dir)
sc.settings.figdir = '.'
print(f"Loaded atlas: {adata_main.n_obs:,} cells × {adata_main.n_vars:,} genes")

Loaded atlas: 2,138,694 cells × 17,411 genes


In [18]:
adata_main.var_names
len(adata_main.var_names)

17411

In [20]:
adata_main.obs.columns

Index(['n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts',
       'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt',
       'log1p_total_counts_mt', 'pct_counts_mt', 'sample_id', 'n_genes',
       'batch', '_scvi_batch', '_scvi_labels', 'leiden1_anno', 'disease',
       'subcluster_anno', 'severity', 'stage'],
      dtype='object')

In [21]:
adata_main.var.columns

Index(['mt', 'n_cells', 'highly_variable', 'means', 'dispersions',
       'dispersions_norm'],
      dtype='object')

# 2. Object and metadata inspection

Inspect the processed atlas structure and required metadata before downstream analysis.

In [10]:
print(adata_main.obs.columns)

Index(['n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts',
       'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt',
       'log1p_total_counts_mt', 'pct_counts_mt', 'sample_id', 'n_genes',
       'batch', '_scvi_batch', '_scvi_labels', 'leiden1_anno', 'disease',
       'subcluster_anno', 'severity', 'stage'],
      dtype='object')


In [26]:
# Check specific columns
for c in adata_main.obs['disease'].unique():
    print(c)

CAR-T_CRS
HC
SLE
COVID19


In [ ]:
# Print QC parameters as violin
sc.pl.violin(adata_main, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'], groupby='disease', rotation=90)

In [19]:
# Create one 'group' column that combines 'stage' and 'severity'
adata_main.obs['group'] = adata_main.obs['stage'].astype(str) + '_' + adata_main.obs['severity'].astype(str)

In [ ]:
# Attach sample type information while preserving cell indices and order
sample_metadata = pd.read_excel(sample_metadata_path)
required_sample_metadata_columns = {'sample_id', 'sample_type'}
missing_sample_metadata_columns = required_sample_metadata_columns - set(sample_metadata.columns)
if missing_sample_metadata_columns:
    raise ValueError(
        f"Sample metadata is missing columns: {sorted(missing_sample_metadata_columns)}"
    )
if sample_metadata['sample_id'].duplicated().any():
    duplicated_sample_ids = (
        sample_metadata.loc[sample_metadata['sample_id'].duplicated(), 'sample_id']
        .astype(str).unique().tolist()
    )
    raise ValueError(f"Duplicated sample IDs in sample metadata: {duplicated_sample_ids[:10]}")

sample_type_map = sample_metadata.set_index('sample_id')['sample_type']
adata_main.obs['sample_type'] = adata_main.obs['sample_id'].map(sample_type_map)
missing_sample_type_ids = (
    adata_main.obs.loc[adata_main.obs['sample_type'].isna(), 'sample_id']
    .astype(str).unique().tolist()
)
if missing_sample_type_ids:
    raise ValueError(f"Missing sample_type metadata for sample IDs: {missing_sample_type_ids[:10]}")

print(adata_main.obs['sample_type'].value_counts(dropna=False))

sample_type
PBMC     1510633
B         270825
CAR-T     266675
T_B        65992
T          24569
Name: count, dtype: int64


# 3. QC and sample-level statistics

## QC and lineage counts
---
Table S1b, Fig. S1B

In [39]:
# QC and cell counts
# Supplementary Table S1b: sample-level QC and lineage counts

# checks
required_cols = [
    "sample_id", "leiden1_anno", "n_genes_by_counts", "total_counts", "pct_counts_mt"
]
missing = [c for c in required_cols if c not in adata_main.obs.columns]
if missing:
    raise ValueError(f"Missing required columns in adata_main.obs: {missing}")

# data preparation
qc_obs = adata_main.obs[required_cols].copy()

# ensure numeric QC metrics
for c in ["n_genes_by_counts", "total_counts", "pct_counts_mt"]:
    qc_obs[c] = pd.to_numeric(qc_obs[c], errors="coerce")

# total cells per sample
cells_after_qc = qc_obs.groupby("sample_id").size().rename("Cells_after_QC")

# medians per sample
qc_medians = qc_obs.groupby("sample_id")[["n_genes_by_counts", "total_counts", "pct_counts_mt"]].median()
qc_medians.columns = ["Median_n_genes", "Median_n_UMIs", "Median_percent_mito"]

# lineage counts per sample
lineages = ["T", "B", "NK", "Plasma", "Myeloid", "pDC"]
lineage_counts = pd.crosstab(qc_obs["sample_id"], qc_obs["leiden1_anno"]).reindex(columns=lineages, fill_value=0)
lineage_counts = lineage_counts.rename(columns={
    "T": "n_T_cells",
    "B": "n_B_cells",
    "NK": "n_NK_cells",
    "Plasma": "n_Plasma_cells",
    "Myeloid": "n_Myeloid_cells",
    "pDC": "n_pDC"
})

# assemble table
table_s1b = (
    pd.concat([cells_after_qc, qc_medians, lineage_counts], axis=1)
    .reset_index()
    .rename(columns={"sample_id": "Sample_id"})
)

# enforce integer counts
count_cols = [
    "Cells_after_QC", "n_T_cells", "n_B_cells", "n_NK_cells",
    "n_Plasma_cells", "n_Myeloid_cells", "n_pDC"
]
table_s1b[count_cols] = table_s1b[count_cols].fillna(0).astype(int)

# excel formatting and output
table_s1b_output_file = "Table_S1b_sample_level_QC_and_cell_counts.xlsx"
with pd.ExcelWriter(table_s1b_output_file, engine="openpyxl") as writer:
    table_s1b.to_excel(writer, sheet_name="Table_S1b", index=False)

    # formatting
    ws = writer.book["Table_S1b"]
    ws.freeze_panes = "A2"
    header_font = Font(bold=True)

    # bold headers
    for cell in ws[1]:
        cell.font = header_font

    # set column widths
    for col in ws.columns:
        max_len = max(len(str(cell.value)) if cell.value is not None else 0 for cell in col)
        col_letter = get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max(12, min(40, max_len + 2))

    # number formatting for Table_S1b
    col_index = {cell.value: cell.column for cell in ws[1]}

    # integer format
    for col_name in count_cols:
        col = col_index.get(col_name)
        if col:
            for row in range(2, ws.max_row + 1):
                ws.cell(row=row, column=col).number_format = "0"

    # two-decimal format for medians
    for col_name in ["Median_n_genes", "Median_n_UMIs", "Median_percent_mito"]:
        col = col_index.get(col_name)
        if col:
            for row in range(2, ws.max_row + 1):
                ws.cell(row=row, column=col).number_format = "0.00"

print(f"Saved: {table_s1b_output_file}")

/tmp/ipykernel_1076123/2155334434.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  cells_after_qc = qc_obs.groupby("sample_id").size().rename("Cells_after_QC")
/tmp/ipykernel_1076123/2155334434.py:23: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  qc_medians = qc_obs.groupby("sample_id")[["n_genes_by_counts", "total_counts", "pct_counts_mt"]].median()


Saved: Table_S1b_sample_level_QC_and_cell_counts.xlsx


# 4. Atlas composition and UMAP visualization

## Sankey plot
---
Fig. 1B

In [40]:
# Remove HC samples (focus on diseases)
adata_no_hc = adata_main[adata_main.obs['disease'] != 'HC'].copy()

In [ ]:
# helper function to convert hex to rgba for transparent flows
def hex_to_rgba(hex_color, opacity=0.8):
    """Helper to convert hex to rgba for transparent flows."""
    hex_color = hex_color.lstrip('#')
    return f"rgba({int(hex_color[0:2], 16)}, {int(hex_color[2:4], 16)}, {int(hex_color[4:6], 16)}, {opacity})"

# function to plot sankey diagram
def plot_fluent_sankey(adata):
    # Data processing
    cols = ['disease', 'stage', 'severity', 'leiden1_anno']
    df = adata.obs[cols].copy()

    # Filter
    df = df[df['disease'] != 'HC']
    df = df[df['stage'] != 'Control']

    # Merge stages
    def merge_stage(val):
        val_str = str(val).lower()
        if 'before' in val_str: return 'Before'
        if 'pro' in val_str: return 'Progression'
        if 'con' in val_str: return 'Convalescence'
        if 'sle' in val_str: return 'Progression'
        return str(val)
    df['stage_mapped'] = df['stage'].apply(merge_stage)

    # Merge severity
    severity_map = {
        'Severe': 'Severe',
        'Moderate': 'Moderate',
        'No_CRS': 'None', 'None': 'None'
    }
    df['severity_mapped'] = df['severity'].map(severity_map).fillna('None')

    # Deliberately give each disease equal total weight so the largest cohort
    # does not dominate the visual display. This is not a raw cell-count Sankey
    disease_counts = df['disease'].value_counts()
    df['weight'] = df['disease'].map(lambda x: 100.0 / disease_counts[x])
    df['weight'] = pd.to_numeric(df['weight'], errors='coerce')
    df = df.dropna(subset=['weight'])

    # Colors and configurations
    # Fixed colors for diseases and immune cells, keep middle stages and severities grey (avoid visual clutter)
    color_map = {
        'CAR-T_CRS': "#1a86d3", 'COVID19': '#ff7f0e', 'SLE': '#2ca02c',
        'Before': '#b0b0b0', 'Progression': '#b0b0b0', 'Convalescence': '#b0b0b0',
        'Severe': '#b0b0b0', 'Moderate': '#b0b0b0', 'None': '#b0b0b0',
        'T': '#5D4599', 'NK': '#AE435A', 'Myeloid': '#EE954F', 
        'B': '#4C987B', 'Plasma': "#9DC7B7", 'pDC': "#ECC67A"
    }

    # Custom order
    custom_orders = {
        'disease': ['CAR-T_CRS', 'COVID19', 'SLE'],
        'stage_mapped': ['Convalescence', 'Progression', 'Before'],
        'severity_mapped': ['Severe', 'Moderate', 'None']
    }
    
    flow_cols = ['disease', 'stage_mapped', 'severity_mapped', 'leiden1_anno']

    # Build nodes
    labels = []
    node_colors = []
    id_registry = {}
    counter = 0

    for col in flow_cols:
        id_registry[col] = {}
        
        # Get unique values
        existing = set(df[col].unique())
        
        # Sort based on custom orders to guide the layout
        if col in custom_orders:
            unique_vals = [x for x in custom_orders[col] if x in existing]
            unique_vals += sorted(list(existing - set(unique_vals)))
        else:
            # For cell types, sorting by abundance
            counts = df[col].value_counts()
            unique_vals = counts.index.tolist()

        for val in unique_vals:
            id_registry[col][val] = counter
            labels.append(val)
            
            # Colors
            c = 'lightgrey'
            if val in color_map:
                c = color_map[val]
            else:
                for k, v in color_map.items():
                    if k.lower() in str(val).lower():
                        c = v
                        break
            node_colors.append(c)
            counter += 1

    # Links, consistent colors with diseases
    sources, targets, values, link_colors = [], [], [], []

    for i in range(len(flow_cols) - 1):
        src_col = flow_cols[i]
        tgt_col = flow_cols[i+1]

        # Prevent duplicate 'disease' column error
        if 'disease' in [src_col, tgt_col]:
            keys = [src_col, tgt_col]
        else:
            keys = [src_col, tgt_col, 'disease']

        grouped = df.groupby(keys)['weight'].sum().reset_index()

        for _, row in grouped.iterrows():
            if src_col == 'disease': dis_val = row[src_col]
            else: dis_val = row['disease']

            sources.append(id_registry[src_col][row[src_col]])
            targets.append(id_registry[tgt_col][row[tgt_col]])
            values.append(row['weight'])
            
            # Link color
            link_c = hex_to_rgba(color_map.get(dis_val, '#999999'), 0.6)
            link_colors.append(link_c)

    # Plotting
    fig = go.Figure(data=[go.Sankey(
        arrangement="snap",  # 'snap' minimizes overlap while respecting input order
        node=dict(
            pad=5,   
            thickness=40,
            line=dict(color="white", width=0),
            label=labels,
            color=node_colors,

        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=link_colors
        )
    )])

    fig.update_layout(
        title_text="",
        font_size=10,
        width=900,     
        height=600, 
        margin=dict(l=20, r=20, t=20, b=20)
    )
    
    return fig

# apply the function to plot the sankey
fig = plot_fluent_sankey(adata_no_hc)
fig.show()

fig.write_image('sankey.pdf')

## Cell counts visualization
---
Fig. S1F

In [ ]:
# Calculate cell counts per group
group_counts = adata_main.obs['group'].value_counts()
df_counts = group_counts.reset_index()
df_counts.columns = ['group', 'count']

# Map disease information to the groups
group_to_disease = adata_main.obs[['group', 'disease']].drop_duplicates().set_index('group')['disease']
df_counts['disease'] = df_counts['group'].map(group_to_disease)

# Sort the data by disease and count
df_counts = df_counts.sort_values(by=['disease', 'count'], ascending=[True, True])

# Apply log10 transformation (for better visualization)
df_counts['log_count'] = np.log10(df_counts['count'])

# Define the palette
cell_count_disease_palette = {
    'CAR-T_CRS': '#1C7DC2',
    'COVID19': '#EE7B20',
    'SLE': '#329839',
    'HC': "#E0E0E0"
}

# Assign colors
colors = df_counts['disease'].map(cell_count_disease_palette).astype(object).fillna('grey')

# Create the plot
fig, ax = plt.subplots(figsize=(10, len(df_counts) * 0.4 + 1), dpi=300)

# Create positions for the lollipops
y_pos = range(len(df_counts))

x_start = 3.8
x_end = 6.2

# Draw the horizontal lines (stems) based on log counts
ax.hlines(y=y_pos, xmin=x_start, xmax=df_counts['log_count'], color=colors, alpha=0.6, linewidth=2)

# Draw the points (lollipops) based on log counts
ax.scatter(df_counts['log_count'], y_pos, color=colors, s=100, alpha=1, zorder=3)

# Add text labels displaying original counts
for i, (count, log_val) in enumerate(zip(df_counts['count'], df_counts['log_count'])):
    ax.text(log_val + (x_end - x_start) * 0.02, i, f"{count:,}", va='center', fontsize=9, color='black')

# Formatting
ax.set_yticks(y_pos)
ax.set_yticklabels(df_counts['group'], fontsize=10)
ax.set_xlabel('Log10(Cell Count)')
ax.set_title('Cell Counts per Group (Log10 Scale)')

# Apply the requested x-limits with margins
ax.set_xlim(x_start, x_end)

# Remove borders
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_visible(True)

# Grid
ax.grid(axis='x', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('group_cell_counts_lollipop_log10.pdf', bbox_inches='tight')
plt.show()

## UMAP

### Major and subcluster UMAPs
---
Fig. 1C

In [ ]:
# Major UMAP with major immune cell lineages 
custom_palette = {
    'T' : '#17becf',
    'B' : '#279e68',
    'Myeloid': '#ff7f0e',
    'Plasma': '#279e68',
    'NK': '#E34234',
    'pDC': '#8C564B'
}

fig, ax = plt.subplots(figsize=(6, 4), dpi=300)
sc.pl.umap(
    adata_main,
    color='leiden1_anno',
    palette=custom_palette,
    size=0.2,
    frameon=False,
    ax=ax,
    show=False
)

plt.tight_layout()
plt.savefig('UMAP_major_celltype.png', dpi=300, bbox_inches='tight')

In [ ]:
# Detailed subcluster UMAP
plt.rcParams['figure.dpi'] = 300

# Define a dictionary to map each subcluster to a specific color
custom_palette = {
    # CD4+ T-cells 
    'Tn_CD4_LEF1': '#B799E9',
    'Tcm_CD4_IL7R': '#A781E2',
    'Tem_CD4_GZMK': '#976ADF',
    'Treg_CD4_FOXP3': '#8651DA',
    'Ta_CD4_MKI67': '#7638D3',
    
    # CD8+ T-cells 
    'Tn_CD8_LEF1': '#4292C6',
    'Tem_CD8_GZMK': '#6BAED6',
    'Tctl_CD8_GZMB': '#9ECAE1',
    'Ta_CD8_MKI67': '#C6DBEF',
    'Tex_CD8_PDCD1': '#DEEBF7',
    
    # NKT cells
    'Tnkl_KLRD1': '#EB7035',
    
    # NK cells
    'NK_TXK': '#E24A1F',
    'NK_CCL3': '#B33E5C',
    'NK_KLRC1': '#A23450',
    
    # Myeloid cells (Monocytes & DCs)
    'Mono_CD14_IL1B': '#F58549',
    'Mono_CD14_S100A8': "#F15806",
    'Mono_CD14_IFI44': "#FF6E4D",
    'Mono_CD14_CD16': '#FCDD60',
    'Mono_CD16_LST1': '#FCE300',
    'cDC_CD1C': '#FFC200',
    'pDC_LILRA4': '#F5B01E',
    
    # B cells
    'Bn_CD19_TCL1A': '#54B98E',
    'Bm_CD19_CD27': '#4C9A7E',
    'Plasma_TNFRSF17': '#61A384'
}

# set figure size
plt.rcParams['figure.figsize'] = (6, 5)

# Plot the UMAP
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
sc.pl.umap(
    adata_main,
    color='subcluster_anno',
    palette=custom_palette,
    size=0.5,
    frameon=False,
    legend_loc='none',
    ax=ax,
    show=False
)
plt.tight_layout()
fig.savefig('UMAP_subclusters.png', dpi=300, bbox_inches='tight')

### CAR-T cells in the main UMAP
---
Fig. S1E

In [ ]:
# Add a new column 'CAR-T_anno' to separate CAR-T cells
# CAR-T cells are annotated from the source by sample IDs (pure CAR-T cells extracted through FACS)
CAR_T_samples = [
    "CART_S005",
    "CART_S006",
    "CART_S007",
    "CART_S012",
    "CART_S013",
    "CART_S014",
    "CART_S019",
    "CART_S020",
    "CART_S021",
    "CART_S026",
    "CART_S027",
    "CART_S028",
    "CART_S033",
    "CART_S034",
    "CART_S035",
    "CART_S040",
    "CART_S041",
    "CART_S042",
    "CART_S046",
    "CART_S047",
    "CART_S048",
    "CART_S053",
    "CART_S054",
    "CART_S055",
    "CART_S060",
    "CART_S061",
    "CART_S062",
    "CART_S067",
    "CART_S068",
    "CART_S069",
    "CART_S074",
    "CART_S075",
    "CART_S076",
    "CART_S081",
    "CART_S082",
    "CART_S083",
    "CART_S087",
    "CART_S088",
    "CART_S089",
    "CART_S094",
    "CART_S095",
    "CART_S096",
    "CART_S101",
    "CART_S102",
    "CART_S103",
    "CART_S108",
    "CART_S109",
    "CART_S110"
]

# Create the CAR-T_anno column by setting all cells to 'non-CAR-T' first
adata_main.obs['CAR-T_anno'] = 'non-CAR-T'
# Then set CAR-T samples to 'CAR-T'
adata_main.obs.loc[adata_main.obs['sample_id'].isin(CAR_T_samples), 'CAR-T_anno'] = 'CAR-T'

# Validate the constructed annotation before downstream comparisons
expected_car_t_annotations = {'CAR-T', 'non-CAR-T'}
observed_car_t_annotations = set(adata_main.obs['CAR-T_anno'].dropna().astype(str))
if observed_car_t_annotations != expected_car_t_annotations:
    raise ValueError(
        'CAR-T_anno must contain exactly CAR-T and non-CAR-T; observed: '
        f"{sorted(observed_car_t_annotations)}"
    )
if not adata_main.obs.loc[adata_main.obs['CAR-T_anno'] == 'CAR-T', 'sample_id'].isin(CAR_T_samples).all():
    raise ValueError('CAR-T_anno contains CAR-T cells outside the approved CAR_T_samples mapping.')

In [ ]:
# UMAP by 'CAR-T_anno', cells other than CAR-T are grey

# Create figure
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)

# Plot background (all cells in lightgrey)
sc.pl.umap(
    adata_main,
    color='CAR-T_anno',
    palette={cat: 'lightgrey' for cat in adata_main.obs['CAR-T_anno'].unique()},
    size=0.5,
    frameon=False,
    ax=ax,
    legend_loc=None,
    show=False
)

# Overlay CAR-T cells
adata_car_t_umap = adata_main[adata_main.obs['CAR-T_anno'] == 'CAR-T']

sc.pl.umap(
    adata_car_t_umap,
    color='CAR-T_anno',
    palette={'CAR-T': "#774FB6"},
    size=1,
    frameon=False,
    ax=ax,
    title='CAR-T Cells',
    legend_loc='none',
    show=False
)

plt.tight_layout()
plt.savefig('UMAP_CAR-T_highlight.png', bbox_inches='tight', dpi=300)
plt.show()

# 5. Integration-quality assessment

## Downsampled UMAP and entropy plots by diseases
---
Fig. S1D

In [ ]:
# Downsampling to show the harmonization

# Number of cells to sample per disease
n_cells_per_disease = 50000

integration_disease_palette = {
    'CAR-T_CRS': '#5DA5DA',
    'COVID19': '#FF9D5C',
    'SLE': '#6FBF73',
    'HC': '#E0E0E0'
}

# get the indices for balanced downsampling
disease_sample_indices = (
    adata_main.obs.groupby('disease')
    .apply(lambda x: x.sample(n=min(len(x), n_cells_per_disease), random_state=RANDOM_SEED))
    .reset_index(level=0, drop=True)
    .index
    .tolist()
)

# Create a downsampled skeleton object to avoid memory explosion
subset_obs = adata_main.obs.loc[disease_sample_indices, :].copy()
subset_umap = adata_main.obsm['X_umap'][adata_main.obs.index.get_indexer(disease_sample_indices)]
subset_pca = adata_main.obsm['X_pca'][adata_main.obs.index.get_indexer(disease_sample_indices)]

# Create the lightweight AnnData object
adata_vis = ad.AnnData(obs=subset_obs)
adata_vis.obsm['X_umap'] = subset_umap
adata_vis.obsm['X_pca'] = subset_pca

# Randomize order so that the plotting order is not biased
disease_plot_order = np.random.default_rng(RANDOM_SEED).permutation(adata_vis.n_obs)
adata_vis = adata_vis[disease_plot_order].copy()

# plot downsampled umap
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
sc.pl.umap(
    adata_vis,
    color='disease',
    palette=integration_disease_palette,
    size=5,
    alpha=1,
    frameon=False,
    legend_loc='right margin',
    ax=ax,
    show=False
)
fig.savefig('UMAP_downsampled_by_disease.png', bbox_inches='tight', dpi=300)


In [ ]:
# Entropy of mixing

# calculate neighbors for adata_vis
sc.pp.neighbors(adata_vis, n_neighbors=30, use_rep='X_pca')

# entropy calculation function
def calculate_per_cell_entropy(adata, label_col='disease'):

    dummies = pd.get_dummies(adata.obs[label_col])
    labels_onehot = csc_matrix(dummies.values)

    connectivities = adata.obsp['connectivities']

    neighbor_counts = connectivities.dot(labels_onehot)

    neighbor_counts = neighbor_counts.toarray()

    probs = neighbor_counts / neighbor_counts.sum(axis=1, keepdims=True)

    n_labels = labels_onehot.shape[1]
    entropy_scores = entropy(probs.T, base=n_labels) # entropy expects (N_labels, N_samples)
    return entropy_scores

print("Calculating per-cell entropy...")
adata_vis.obs['Mixing_Entropy'] = calculate_per_cell_entropy(adata_vis, 'disease')
print("Done.")

# visualization of mixing entropy 
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
sc.pl.umap(
    adata_vis,
    color='Mixing_Entropy',
    cmap='Blues',
    size=3,
    frameon=False,
    legend_loc='none',
    ax=ax,
    show=False
)
fig.savefig('UMAP_mixing_entropy.png', bbox_inches='tight', dpi=300)


## Downsampled UMAP by severity
---
Fig. S1G

In [ ]:
# Create a temporary column combining disease and severity
adata_main.obs['ds'] = adata_main.obs['disease'].astype(str) + '_' + adata_main.obs['severity'].astype(str)

# Downsampling by group to show the harmonization

# Number of cells to sample per group
n_cells_per_group = 50000

# Define a custom palette with shades based on severity
# Disease > Color Class; Severity > Shade (Darker = Severe)
severity_group_palette = {
    # CAR-T_CRS: Blue shades
    'CAR-T_CRS_Severe': '#08306b',
    'CAR-T_CRS_Moderate': '#2171b5',
    'CAR-T_CRS_No_CRS': '#6baed6',
    
    # COVID19: Orange shades
    'COVID19_Severe': "#cb754d",
    'COVID19_Moderate': '#fd8d3c',
    
    # SLE: Green shades
    'SLE_Severe': '#006d2c',
    'SLE_Moderate': '#74c476',
    'SLE_nan': '#c7e9c0',
    
    # HC: Grey
    'HC_HC': '#bdbdbd'
}

# get the indices for balanced downsampling
severity_sample_indices = (
    adata_main.obs.groupby('ds')
    .apply(lambda x: x.sample(n=min(len(x), n_cells_per_group), random_state=RANDOM_SEED))
    .reset_index(level=0, drop=True)
    .index
    .tolist()
)

# Create a downsampled skeleton object to avoid memory explosion
subset_ds_obs = adata_main.obs.loc[severity_sample_indices, :].copy()
subset_ds_umap = adata_main.obsm['X_umap'][adata_main.obs.index.get_indexer(severity_sample_indices)]

# Create the lightweight AnnData object
adata_vis_ds = ad.AnnData(obs=subset_ds_obs)
adata_vis_ds.obsm['X_umap'] = subset_ds_umap

# Randomize order so that the plotting order is not biased 
severity_plot_order = np.random.default_rng(RANDOM_SEED).permutation(adata_vis_ds.n_obs)
adata_vis_ds = adata_vis_ds[severity_plot_order].copy()

# Remove SLE_nan for plotting
adata_vis_ds = adata_vis_ds[adata_vis_ds.obs['ds'] != 'SLE_nan'].copy()

# plot umap
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
sc.pl.umap(
    adata_vis_ds,
    color='ds',
    palette=severity_group_palette,
    size=3,
    alpha=1,
    frameon=False,
    legend_loc='right margin',
    ax=ax,
    show=False
)
fig.savefig('UMAP_downsampled_by_severity.png', bbox_inches='tight', dpi=300)

## Downsampled UMAP by stages
---
Fig. S1G

In [ ]:
# Downsampling by group to show the harmonization, avoiding overplotting

# Number of cells to sample per group
n_cells_per_group = 50000

# Define a custom palette with shades based on stages
stage_group_palette = {
    # CAR-T_CRS: Blue shades
    'CAR-T_CRS_pro': '#08306b',   # Dark Blue
    'CAR-T_CRS_con': '#2171b5', # Medium Blue
    'CAR-T_CRS_before': '#6baed6',   # Light Blue
    
    # COVID19: Orange shades
    'COVID19_pro': "#cb754d",     # Dark Orange
    'COVID19_con': '#fd8d3c',   # Medium Orange
    
    # SLE: Green shades
    'SLE': '#74c476',       # Medium Green

    # HC: Grey
    'HC': '#bdbdbd'
}

# get the indices for balanced downsampling
stage_sample_indices = (
    adata_main.obs.groupby('stage')
    .apply(lambda x: x.sample(n=min(len(x), n_cells_per_group), random_state=RANDOM_SEED))
    .reset_index(level=0, drop=True)
    .index
    .tolist()
)

# Create a downsampled skeleton object to avoid memory explosion
subset_obs_dst = adata_main.obs.loc[stage_sample_indices, :].copy()
subset_umap_dst = adata_main.obsm['X_umap'][adata_main.obs.index.get_indexer(stage_sample_indices)]

# Create the lightweight AnnData object
adata_vis_dst = ad.AnnData(obs=subset_obs_dst)
adata_vis_dst.obsm['X_umap'] = subset_umap_dst

# Randomize order so that the plotting order is not biased 
stage_plot_order = np.random.default_rng(RANDOM_SEED).permutation(adata_vis_dst.n_obs)
adata_vis_dst = adata_vis_dst[stage_plot_order].copy()

# plot umap
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
sc.pl.umap(
    adata_vis_dst,
    color='stage',
    palette=stage_group_palette,
    size=3,
    alpha=1,
    frameon=False,
    legend_loc='right margin',
    ax=ax,
    show=False
)
fig.savefig('UMAP_downsampled_by_stage.png', bbox_inches='tight', dpi=300)

# 6. Subcluster DEG and markers
---
Table S1c, Fig. S1I

In [53]:
# Top marker genes for each subcluster (Table S1c)

import time
from collections import Counter
from datetime import datetime
from openpyxl import load_workbook

subcluster_marker_output_xlsx = Path("Table_S1c_top50_subcluster_marker_genes.xlsx")
subcluster_marker_output_csv = Path("Table_S1c_top50_subcluster_marker_genes.csv")
subcluster_marker_validation_csv = Path("Table_S1c_subcluster_marker_validation_summary.csv")

SUBCLUSTER_COL = "subcluster_anno"
LINEAGE_COL = "leiden1_anno"

MAX_CELLS_PER_SUBCLUSTER = 5000
MIN_PCT = 0.05
N_TOP_GENES = 3000
MARKERS_PER_SUBCLUSTER = 50
SUBCLUSTER_DEG_RANDOM_SEED = RANDOM_SEED


def log_progress(message):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}")


def matrix_appears_log_normalized(x):
    try:
        max_value = x.max() if sparse.issparse(x) else np.max(x)
        if hasattr(max_value, "A"):
            max_value = max_value.A.item()
        return float(max_value) <= 50
    except Exception:
        return False


def compute_gene_detection_fraction(x_mat, gene_idx):
    if len(gene_idx) == 0:
        return np.array([])
    expr = (x_mat[:, gene_idx] > 0).mean(axis=0)
    if sparse.issparse(expr):
        return expr.A1
    return np.asarray(expr).ravel()


def format_marker_workbook(path_xlsx):
    wb = load_workbook(path_xlsx)
    for ws in wb.worksheets:
        ws.freeze_panes = "A2"
        for cell in ws[1]:
            cell.font = Font(bold=True)
        for col in ws.columns:
            max_len = 0
            col_letter = col[0].column_letter
            for cell in col:
                val = "" if cell.value is None else str(cell.value)
                max_len = max(max_len, len(val))
            ws.column_dimensions[col_letter].width = max(10, min(50, max_len + 2))
        if ws.title == "Table_S1c":
            for cell in ws["E"]:
                if cell.row > 1:
                    cell.number_format = "0.000"
            for cell in ws["F"]:
                if cell.row > 1:
                    cell.number_format = "0.00E+00"
    wb.save(path_xlsx)


start_time = time.time()

for required_col in [SUBCLUSTER_COL, LINEAGE_COL]:
    if required_col not in adata_main.obs.columns:
        raise ValueError(f"Missing required metadata column: {required_col}")

if not adata_main.obs_names.is_unique:
    raise ValueError("adata_main.obs_names must be unique before subcluster DEG analysis.")
if not adata_main.var_names.is_unique:
    raise ValueError("adata_main.var_names must be unique before subcluster DEG analysis.")

log_progress(
    f"adata_main.X appears log-normalized: {matrix_appears_log_normalized(adata_main.X)}"
)

if "highly_variable" in adata_main.var.columns:
    hvg_mask = adata_main.var["highly_variable"].astype(bool).to_numpy()
    log_progress("Using existing highly_variable genes from adata_main.var.")
else:
    log_progress("Computing highly variable genes on a temporary copy.")
    adata_hvg = adata_main.copy()
    sc.pp.highly_variable_genes(
        adata_hvg,
        n_top_genes=N_TOP_GENES,
        flavor="seurat_v3"
    )
    hvg_mask = adata_hvg.var["highly_variable"].astype(bool).to_numpy()

hvg_genes = adata_main.var_names[hvg_mask]
if len(hvg_genes) == 0:
    raise ValueError("No highly variable genes are available for subcluster DEG analysis.")
log_progress(f"Number of HVGs used: {len(hvg_genes):,}")

subclusters = adata_main.obs[SUBCLUSTER_COL].astype(str)
subcluster_categories = subclusters.dropna().unique().tolist()
orig_counts = subclusters.value_counts().to_dict()

log_progress("Downsampling cells within each subcluster for marker detection.")
rng = np.random.default_rng(SUBCLUSTER_DEG_RANDOM_SEED)
selected_idx = []
down_counts = {}
for subcluster_name in subcluster_categories:
    idx = np.flatnonzero(subclusters.to_numpy() == subcluster_name)
    if len(idx) > MAX_CELLS_PER_SUBCLUSTER:
        chosen = rng.choice(idx, size=MAX_CELLS_PER_SUBCLUSTER, replace=False)
    else:
        chosen = idx
    selected_idx.append(chosen)
    down_counts[subcluster_name] = len(chosen)
    log_progress(
        f"{subcluster_name}: original={len(idx):,}, downsampled={len(chosen):,}"
    )

selected_idx = np.concatenate(selected_idx)

log_progress("Creating downsampled AnnData restricted to HVGs.")
adata_subcluster_markers = adata_main[selected_idx, :][:, hvg_genes].copy()

log_progress("Running subcluster marker detection with Wilcoxon tests.")
sc.tl.rank_genes_groups(
    adata_subcluster_markers,
    groupby=SUBCLUSTER_COL,
    method="wilcoxon",
    n_genes=len(hvg_genes),
    use_raw=False,
)

log_progress("Determining major lineage per subcluster from the full atlas object.")
major_lineage_map = {}
for subcluster_name in subcluster_categories:
    vals = adata_main.obs.loc[
        subclusters.to_numpy() == subcluster_name, LINEAGE_COL
    ].astype(str)
    major_lineage_map[subcluster_name] = (
        Counter(vals).most_common(1)[0][0] if len(vals) else ""
    )

log_progress("Filtering and ranking marker genes.")
results_rows = []
summary_rows = []
gene_to_idx = {gene: i for i, gene in enumerate(adata_subcluster_markers.var_names)}
x_all = adata_subcluster_markers.X
subclusters_down = adata_subcluster_markers.obs[SUBCLUSTER_COL].astype(str).to_numpy()

for subcluster_name in subcluster_categories:
    df = sc.get.rank_genes_groups_df(adata_subcluster_markers, group=subcluster_name)
    if df is None or df.empty:
        log_progress(f"Warning: no DE results for {subcluster_name}")
        continue

    df = df.rename(columns={
        "names": "Gene",
        "logfoldchanges": "Log2FC",
        "pvals_adj": "Adjusted_p_value",
        "pvals": "P_value",
    })
    if "Adjusted_p_value" not in df.columns or df["Adjusted_p_value"].isna().all():
        df["Adjusted_p_value"] = df.get("P_value", np.nan)

    genes = df["Gene"].astype(str).tolist()
    retained_genes_for_pct = [gene for gene in genes if gene in gene_to_idx]
    gene_idx = [gene_to_idx[gene] for gene in retained_genes_for_pct]

    in_mask = subclusters_down == subcluster_name
    out_mask = ~in_mask

    pct_in = compute_gene_detection_fraction(x_all[in_mask, :], gene_idx)
    pct_out = compute_gene_detection_fraction(x_all[out_mask, :], gene_idx)
    pct_map = {
        gene: (pct_in[i], pct_out[i])
        for i, gene in enumerate(retained_genes_for_pct)
    }

    df["Pct_in_subcluster"] = df["Gene"].map(lambda gene: pct_map.get(gene, (0.0, 0.0))[0])
    df["Pct_out_subcluster"] = df["Gene"].map(lambda gene: pct_map.get(gene, (0.0, 0.0))[1])

    base_filter = (
        (df["Pct_in_subcluster"] >= MIN_PCT)
        & (df["Log2FC"] > 0)
        & (df["Adjusted_p_value"] < 0.05)
    )
    df_base = df[base_filter].copy()
    df_strict = df_base[df_base["Log2FC"] > 0.5].copy()
    df_filtered = df_strict if len(df_strict) >= MARKERS_PER_SUBCLUSTER else df_base
    df_filtered = df_filtered.sort_values(
        by=["Log2FC", "Adjusted_p_value"], ascending=[False, True]
    )

    retained = df_filtered.head(MARKERS_PER_SUBCLUSTER).copy()
    retained["Rank"] = np.arange(1, len(retained) + 1)
    retained["Subcluster"] = subcluster_name
    retained["Major_lineage"] = major_lineage_map.get(subcluster_name, "")

    results_rows.append(
        retained[[
            "Subcluster",
            "Major_lineage",
            "Rank",
            "Gene",
            "Log2FC",
            "Adjusted_p_value",
        ]]
    )

    summary_rows.append({
        "Subcluster": subcluster_name,
        "Major_lineage": major_lineage_map.get(subcluster_name, ""),
        "Original_cell_count": int(orig_counts.get(subcluster_name, 0)),
        "Downsampled_cell_count": int(down_counts.get(subcluster_name, 0)),
        "Number_HVGs_used": int(len(hvg_genes)),
        "Number_significant_markers_after_filtering": int(len(df_filtered)),
        "Number_retained_markers": int(len(retained)),
    })

if not results_rows:
    raise ValueError("No marker genes were retained for any subcluster.")

subcluster_marker_df = pd.concat(results_rows, axis=0, ignore_index=True)
subcluster_marker_validation_df = pd.DataFrame(summary_rows)

log_progress("Saving subcluster marker outputs.")
with pd.ExcelWriter(subcluster_marker_output_xlsx, engine="openpyxl") as writer:
    subcluster_marker_df.to_excel(writer, sheet_name="Table_S1c", index=False)
    subcluster_marker_validation_df.to_excel(
        writer, sheet_name="Validation_summary", index=False
    )

subcluster_marker_df.to_csv(subcluster_marker_output_csv, index=False)
subcluster_marker_validation_df.to_csv(subcluster_marker_validation_csv, index=False)
format_marker_workbook(subcluster_marker_output_xlsx)

runtime_seconds = time.time() - start_time
log_progress("Subcluster marker detection complete.")
print(f"Excel: {subcluster_marker_output_xlsx.resolve()}")
print(f"CSV: {subcluster_marker_output_csv.resolve()}")
print(f"Validation CSV: {subcluster_marker_validation_csv.resolve()}")
print(f"Runtime: {runtime_seconds:.2f} seconds")
print(f"Subclusters processed: {subcluster_marker_validation_df.shape[0]}")
print(f"Marker rows exported: {subcluster_marker_df.shape[0]}")
display(subcluster_marker_validation_df)

[2026-06-29 15:18:59] adata_main.X appears log-normalized: True
[2026-06-29 15:18:59] Using existing highly_variable genes from adata_main.var.
[2026-06-29 15:18:59] Number of HVGs used: 1,018
[2026-06-29 15:19:00] Downsampling cells within each subcluster for marker detection.
[2026-06-29 15:19:00] Mono_CD16_LST1: original=45,076, downsampled=5,000
[2026-06-29 15:19:00] Tnkl_KLRD1: original=140,799, downsampled=5,000
[2026-06-29 15:19:00] Mono_CD14_S100A8: original=151,879, downsampled=5,000
[2026-06-29 15:19:00] Tctl_CD8_GZMB: original=158,595, downsampled=5,000
[2026-06-29 15:19:00] Mono_CD14_IL1B: original=134,643, downsampled=5,000
[2026-06-29 15:19:00] Ta_CD8_MKI67: original=91,822, downsampled=5,000
[2026-06-29 15:19:00] Treg_CD4_FOXP3: original=42,477, downsampled=5,000
[2026-06-29 15:19:00] Tcm_CD4_IL7R: original=179,235, downsampled=5,000
[2026-06-29 15:19:00] Tex_CD8_PDCD1: original=28,563, downsampled=5,000
[2026-06-29 15:19:00] Tem_CD8_GZMK: original=172,349, downsampled=5

/home/haikun/miniforge/envs/multics/lib/python3.11/site-packages/scanpy/tools/_rank_genes_groups.py:456: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/home/haikun/miniforge/envs/multics/lib/python3.11/site-packages/scanpy/tools/_rank_genes_groups.py:458: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/home/haikun/miniforge/envs/multics/lib/python3.11/site-packages/scanpy/tools/_rank_genes_groups.py:461: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which ha

[2026-06-29 15:19:23] Determining major lineage per subcluster from the full atlas object.
[2026-06-29 15:19:24] Filtering and ranking marker genes.
[2026-06-29 15:19:52] Saving subcluster marker outputs.
[2026-06-29 15:19:52] Subcluster marker detection complete.
Excel: /home/haikun/MultiCS_hk/outputs/global_atlas/Table_S1c_top50_subcluster_marker_genes.xlsx
CSV: /home/haikun/MultiCS_hk/outputs/global_atlas/Table_S1c_top50_subcluster_marker_genes.csv
Validation CSV: /home/haikun/MultiCS_hk/outputs/global_atlas/Table_S1c_subcluster_marker_validation_summary.csv
Runtime: 71.58 seconds
Subclusters processed: 24
Marker rows exported: 1200


,Subcluster,Major_lineage,Original_cell_count,Downsampled_cell_count,Number_HVGs_used,Number_significant_markers_after_filtering,Number_retained_markers
0,Mono_CD16_LST1,Myeloid,45076,5000,1018,295,50
1,Tnkl_KLRD1,T,140799,5000,1018,150,50
2,Mono_CD14_S100A8,Myeloid,151879,5000,1018,322,50
3,Tctl_CD8_GZMB,T,158595,5000,1018,133,50
4,Mono_CD14_IL1B,Myeloid,134643,5000,1018,369,50
5,Ta_CD8_MKI67,T,91822,5000,1018,140,50
6,Treg_CD4_FOXP3,T,42477,5000,1018,143,50
7,Tcm_CD4_IL7R,T,179235,5000,1018,143,50
8,Tex_CD8_PDCD1,T,28563,5000,1018,190,50
9,Tem_CD8_GZMK,T,172349,5000,1018,118,50


In [ ]:
# subcluster markers
markers = [
    # T cells
    'CD3D', 'CD4', 'CD8A', 'CD8B', 'LEF1', 'CCR7', 'IL7R', 'TCF7', 'SELL', 'S100A4', 'FOXP3', 'MKI67', 'NKG7', 'GZMB', 'PRF1', 'GZMK', 'PDCD1', 'CTLA4', 'LAG3', 'HAVCR2', 'KLRD1',
    # Myeloid cells
    'CST3', 'LYZ', 'FCGR3A','LST1', 'CD14', 'IL1B', 'S100A8', 'S100A9', 'IFI44', 'LGALS3BP', 'CD1C', 'FCER1A',
    # NK and pDC cells
    'NCAM1', 'FCGR3A', 'KLRD1', 'NKG7', 'TXK', 'CCL3', 'KLRC1', 'LILRA4', 'CLEC4C', 'IL3RA',
    # B and Plasma cells
    'CD19', 'MS4A1', 'TCL1A', 'FCER2', 'CD27', 'TNFRSF17', 'MZB1', 'SDC1'
]

# Subcluster order
order = [
    'Tn_CD4_LEF1', 'Tcm_CD4_IL7R', 'Tem_CD4_GZMK', 'Treg_CD4_FOXP3', 'Ta_CD4_MKI67',
    'Tn_CD8_LEF1', 'Tem_CD8_GZMK', 'Tctl_CD8_GZMB', 'Ta_CD8_MKI67', 'Tex_CD8_PDCD1',
    'Tnkl_KLRD1',
    'Mono_CD16_LST1', 'Mono_CD14_CD16', 'Mono_CD14_IL1B', 'Mono_CD14_S100A8', 'Mono_CD14_IFI44',
    'cDC_CD1C',
    'NK_TXK', 'NK_CCL3', 'NK_KLRC1',
    'pDC_LILRA4',
    'Bn_CD19_TCL1A', 'Bm_CD19_CD27', 'Plasma_TNFRSF17'
]

# Create a heatmap for marker gene expression by subclusters
marker_plot = sc.pl.matrixplot(
    adata_main,
    var_names=markers,           # marker gene list
    groupby="subcluster_anno",     # group by subcluster
    categories_order=order,      # specified subcluster order
    use_raw=False,
    cmap='Reds',
    standard_scale='var',
    figsize=(10, 6),
    return_fig=True
)
marker_plot.savefig('subcluster_marker_matrixplot.pdf', bbox_inches='tight')

# 7. Cytokine-storm gene expression and scoring

## Curated CS gene list

In [13]:
# Curated CS gene list (n=60, Table S2a)
cs_genes = [
    # Classical cytokines (both pro-inflammatory and regulatory cytokines)
    'IL1A', 'IL1B', 'IL6', 'TNF', 'IFNG', 'IL10', 'IL2', 'IL18', 'IFNB1', 'CSF2', 'CSF3',
    # Key receptors for key cytokines
    'IL1R1', 'IL6R', 'IL6ST', 'TNFRSF1A', 'TNFRSF1B','CSF2RA', 'CSF3R', 'IFNAR1', 'IFNAR2',
    # Key chemokines and their receptors
    'CCL2', 'CCR2', 'CCL5', 'CCR5', 'CXCR1', 'CXCR2', 'CXCL1',
    # Cytokine-induced/related signaling and regulatory molecules
    'NFKB1', 'NFKBIA', 'NFKBIZ', 'STAT1', 'STAT3', 'SOCS3', 'IRF4', 'JUN', 'FOS', 'IFI27',
    'TNFAIP6', 'CEBPB', 'PTGS2', 
    # Innate immune hyperactivation markers
    'S100A8', 'S100A9', 'S100A12', 'NLRP3', 'CASP1', 'GSDMD', 'FASLG', 'TLR2',
    # Adaptive immune hyperactivation/exhaustion markers
    'PDCD1', 'CTLA4', 'LAG3', 'CD38', 'XBP1', 'PRDM1', 'IL2RA', 'CD40LG', 'ICOS', 'TNFSF13B', 'TNFRSF4', 'TNFRSF9'
]

def require_complete_gene_set(adata, genes, gene_set_name):
    missing = [gene for gene in genes if gene not in adata.var_names]
    if missing:
        raise ValueError(f"Missing genes in {gene_set_name}: {missing}")
    return list(genes)

cs_genes = require_complete_gene_set(adata_main, cs_genes, 'curated CS gene list')

### CS gene expression
---
Table S2d

In [56]:
# Calculate mean expression of cs_genes across 'Disease_Stage_Severity'
# Generate a xlsx file with these columns: CS_gene, Disease, Stage, Severity, Sample_count, Mean_expression, z_score

adata_expr = adata_main

sample_col = "sample_id"
disease_col = "disease"
stage_col = "stage"
severity_col = "severity"

cs_expression_output_file = Path("CS_gene_mean_expression_by_disease_stage_severity.xlsx")

# The complete final CS gene set was validated in the preceding section
analysis_cs_genes = cs_genes
print(f"Number of CS genes used: {len(analysis_cs_genes)}")

# extract expression matrix for the CS genes
X = adata_expr[:, analysis_cs_genes].X

if sparse.issparse(X):
    X = X.tocsr()
else:
    X = np.asarray(X)

# sample-level mean expression calculation
sample_ids = adata_expr.obs[sample_col].astype("category")
sample_categories = sample_ids.cat.categories
sample_codes = sample_ids.cat.codes.to_numpy()

n_cells = adata_expr.n_obs
n_samples = len(sample_categories)

rows = np.arange(n_cells)
cols = sample_codes
data = np.ones(n_cells, dtype=np.float32)

# cells × samples
sample_onehot = csr_matrix((data, (rows, cols)), shape=(n_cells, n_samples))

# samples × genes
sample_sums = sample_onehot.T @ X

sample_counts = np.asarray(sample_onehot.sum(axis=0)).ravel()

if sparse.issparse(sample_sums):
    sample_means = sample_sums.multiply(1 / sample_counts[:, None])
    sample_means = sample_means.toarray()
else:
    sample_means = sample_sums / sample_counts[:, None]

sample_mean_df = pd.DataFrame(
    sample_means,
    index=sample_categories,
    columns=analysis_cs_genes
)

sample_mean_df.index.name = sample_col

# build sample metadata
sample_meta = (
    adata_expr.obs[[sample_col, disease_col, stage_col, severity_col]]
    .drop_duplicates(subset=sample_col)
    .set_index(sample_col)
)

# check one metadata row per sample
if sample_meta.index.duplicated().any():
    duplicated_samples = sample_meta.index[sample_meta.index.duplicated()].unique().tolist()
    raise ValueError(f"Duplicated sample metadata found for: {duplicated_samples[:10]}")

sample_level_gene = sample_meta.join(sample_mean_df, how="inner").reset_index()

print(f"Sample-level table shape: {sample_level_gene.shape}")
print(f"Number of samples retained: {sample_level_gene[sample_col].nunique()}")

# aggregate to disease-stage-severity
group_cols = [disease_col, stage_col, severity_col]

mean_by_group = (
    sample_level_gene
    .groupby(group_cols, observed=True)[analysis_cs_genes]
    .mean()
    .reset_index()
)

sample_count = (
    sample_level_gene
    .groupby(group_cols, observed=True)[sample_col]
    .nunique()
    .rename("Sample_count")
    .reset_index()
)

summary = mean_by_group.merge(sample_count, on=group_cols, how="left")

# long format
long_df = summary.melt(
    id_vars=group_cols + ["Sample_count"],
    value_vars=analysis_cs_genes,
    var_name="CS_gene",
    value_name="Mean_expression"
)

# z score per gene across groups
def zscore(x):
    sd = x.std(ddof=1)
    if sd == 0 or np.isnan(sd):
        return pd.Series(np.zeros(len(x)), index=x.index)
    return (x - x.mean()) / sd

long_df["z_score"] = (
    long_df
    .groupby("CS_gene", group_keys=False)["Mean_expression"]
    .apply(zscore)
)

# rename colnames for output
long_df = long_df.rename(columns={
    disease_col: "Disease",
    stage_col: "Stage",
    severity_col: "Severity"
})

long_df = long_df[
    [
        "CS_gene",
        "Disease",
        "Stage",
        "Severity",
        "Sample_count",
        "Mean_expression",
        "z_score"
    ]
]

# sort
long_df = long_df.sort_values(
    ["CS_gene", "Disease", "Stage", "Severity"]
).reset_index(drop=True)

# save
with pd.ExcelWriter(cs_expression_output_file, engine="openpyxl") as writer:
    long_df.to_excel(writer, index=False, sheet_name="CS_gene_expression_summary")

print(f"Saved: {cs_expression_output_file.resolve()}")
print(long_df.head())

Number of CS genes used: 60
Sample-level table shape: (411, 64)
Number of samples retained: 411
Saved: /home/haikun/MultiCS_hk/outputs/global_atlas/CS_gene_mean_expression_by_disease_stage_severity.xlsx
  CS_gene    Disease             Stage  Severity  Sample_count  \
0   CASP1  CAR-T_CRS  CAR-T_CRS_before  Moderate            18   
1   CASP1  CAR-T_CRS  CAR-T_CRS_before    No_CRS            14   
2   CASP1  CAR-T_CRS  CAR-T_CRS_before    Severe            15   
3   CASP1  CAR-T_CRS     CAR-T_CRS_con  Moderate            12   
4   CASP1  CAR-T_CRS     CAR-T_CRS_con    No_CRS            10   

   Mean_expression   z_score  
0         0.528889 -0.055025  
1         0.529092 -0.051831  
2         0.620884  1.392556  
3         0.454958 -1.218355  
4         0.500955 -0.494575  


### CS gene expression heatmap
---
Fig. 1E

In [57]:
expr_df = adata_main[:, cs_genes].to_df()

# add the group information and calculate the mean expression per group
expr_df['group'] = adata_main.obs['group'].values
cs_expr_df = expr_df.groupby('group').mean()

# compute z-scores
cs_expr_z = cs_expr_df.apply(lambda x: (x - x.mean()) / x.std(ddof=1), axis=0)

/tmp/ipykernel_1076123/2810286630.py:5: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [ ]:
# Heatmap of CS gene expression across groups

# Define the group order as specified
group_order = [
    'CAR-T_CRS_pro_Moderate', 'CAR-T_CRS_pro_Severe',
    'COVID19_pro_Moderate', 'COVID19_pro_Severe',
    'SLE_Moderate', 'SLE_Severe', 'HC_HC'
]

# Reorder rows (groups) and then transpose so that columns are groups and rows are genes
cs_expr_z_ordered = cs_expr_z.loc[group_order].T

# Sort genes
# Identify the group (column index) where each gene has the maximum expression
peak_group_idx = np.argmax(cs_expr_z_ordered.values, axis=1)

# Create a temporary dataframe to help with sorting
# Sort primarily by the peak group index to group genes by their dominant condition
# Secondarily, sort by the peak value to order genes within each block
sort_df = pd.DataFrame({
    'peak_idx': peak_group_idx,
    'peak_val': cs_expr_z_ordered.max(axis=1)
}, index=cs_expr_z_ordered.index)

# Sort: Group index ascending, then peak value descending
sorted_genes = sort_df.sort_values(by=['peak_idx', 'peak_val'], ascending=[True, False]).index

# Reorder the expression matrix
cs_expr_z_sorted = cs_expr_z_ordered.loc[sorted_genes]

# Plotting
plt.figure(figsize=(6, 10)) # Adjusted height to accommodate all genes clearly

ax = sns.heatmap(
    cs_expr_z_sorted,
    center=0,
    cmap='RdBu_r',
    annot=False,
    cbar=False,
    linewidths=0,
    linecolor='white',
    xticklabels=True,
    yticklabels=True
)

ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
ax.set_xlabel("")
ax.set_ylabel("")

# Remove grid
ax.grid(False)

plt.tight_layout()

# Save the figure
plt.savefig('CS_gene_expression_heatmap.pdf', bbox_inches='tight')

plt.show()

### CS gene GO enrichment
---
Fig. 1E

In [59]:
# Subset cs_genes to different categories for separated GO analysis
# categories determined by the heatmap in Fig. 1E
cs_genes_dict = {
    'CAR-T': ['CSF2', 'CD38', 'STAT3', 'CXCR1', 'IL2', 'TNFRSF9', 'NFKB1', 'FASLG', 'TNFRSF1B', 'ICOS', 'NFKBIA', 'NFKBIZ',
            'IL10', 'IFNG', 'TNF', 'CASP1', 'GSDMD', 'CCR5', 'LAG3', 'TNFRSF1A', 'PRDM1', 'XBP1', 'SOCS3',  'STAT1', 'IL2RA', 'PDCD1',
            'CTLA4', 'TNFRSF4', 'IRF4', 'CCL5'],
    'COVID': ['CXCR2', 'IL6', 'CSF3', 'IL1A', 'IL1R1', 'TNFAIP6', 'IFNB1', 'IL6ST', 'IL18', 'CCR2', 'NLRP3', 'PTGS2', 'S100A12',
              'TLR2', 'CSF3R', 'TNFSF13B', 'IL6R', 'IFNAR1', 'CSF2RA', 'IL1B'],
    'SLE': ['FOS', 'CXCL1', 'JUN', 'CCL2', 'IFI27', 'IFNAR2', 'S100A9', 'S100A8', 'CEBPB', 'CD40LG']
}
go_category_genes = [gene for genes in cs_genes_dict.values() for gene in genes]
if len(go_category_genes) != len(set(go_category_genes)):
    raise ValueError('CS GO categories contain duplicated genes.')
if set(go_category_genes) != set(cs_genes):
    raise ValueError('CS GO categories must contain exactly the final curated CS gene list.')

# Perform GO enrichment for each category
go_results_complete = {}
go_results_significant = {}
for category, genes in cs_genes_dict.items():
    category_genes = require_complete_gene_set(
        adata_main, genes, f"{category} CS enrichment gene set"
    )
    
    # Perform GO enrichment using gseapy
    enr = gp.enrichr(
        gene_list=category_genes,
        gene_sets=['GO_Biological_Process_2023'],
        organism='Human',
        outdir=None,
        cutoff=0.05  # Adjusted p-value cutoff
    )
    
    go_results_complete[category] = enr.results.copy()
    go_results_significant[category] = enr.results.loc[
        enr.results['Adjusted P-value'] < 0.05
    ].copy()

# Save both complete enrichment results and the significant subset used downstream.
for category in cs_genes_dict:
    go_results_complete[category].to_csv(
        f'GO_enrichment_{category}_complete.csv', index=False
    )
    go_results_significant[category].to_csv(
        f'GO_enrichment_{category}_significant.csv', index=False
    )

### CS gene expression dotplot
---
Fig. S1J

In [60]:
# Only plot pro_severe and HC samples for clarity
# Create a mask for samples that are either HC or satisfy the 'pro_severe' criteria (including SLE severe)
mask = (
    # Include HC samples
    (adata_main.obs['stage'] == 'HC') | 
    # Include non-SLE samples that are 'pro' and 'Severe'
    ((adata_main.obs['stage'].str.contains('pro', na=False)) & (adata_main.obs['severity'] == 'Severe')) |
    # Include SLE samples that are 'Severe' (regardless of 'pro' in stage)
    ((adata_main.obs['disease'] == 'SLE') & (adata_main.obs['severity'] == 'Severe'))
)

adata_ps = adata_main[mask].copy()

In [ ]:
# Organize the same final CS genes into functional groups for plotting
cs_dotplot_gene_groups = {
    'Cytokines': ['IL1A', 'IL1B', 'IL6', 'TNF', 'IFNG', 'IL10', 'IL2', 'IL18', 'IFNB1', 'CSF2', 'CSF3'],
    'Receptors': ['IL1R1', 'IL6R', 'IL6ST', 'TNFRSF1A', 'TNFRSF1B', 'CSF2RA', 'CSF3R', 'IFNAR1', 'IFNAR2'],
    'Chemokines': ['CCL2', 'CCR2', 'CCL5', 'CCR5', 'CXCR1', 'CXCR2', 'CXCL1'],
    'Signaling': ['NFKB1', 'NFKBIA', 'NFKBIZ', 'STAT1', 'STAT3', 'SOCS3', 'IRF4', 'JUN', 'FOS', 'IFI27', 'TNFAIP6', 'CEBPB', 'PTGS2'],
    'Innate Markers': ['S100A8', 'S100A9', 'S100A12', 'NLRP3', 'CASP1', 'GSDMD', 'FASLG', 'TLR2'],
    'Adaptive Markers': ['PDCD1', 'CTLA4', 'LAG3', 'CD38', 'XBP1', 'PRDM1', 'IL2RA', 'CD40LG', 'ICOS', 'TNFSF13B', 'TNFRSF4', 'TNFRSF9']
}
dotplot_genes = [gene for genes in cs_dotplot_gene_groups.values() for gene in genes]
if len(dotplot_genes) != len(set(dotplot_genes)):
    raise ValueError('CS dotplot groups contain duplicated genes.')
if set(dotplot_genes) != set(cs_genes):
    raise ValueError('CS dotplot groups must contain exactly the final curated CS gene list.')

# Generate the dotplot
dp = sc.pl.dotplot(
    adata_ps,
    var_names=cs_dotplot_gene_groups,
    groupby='group',
    use_raw=False,
    cmap='OrRd',
    standard_scale='var',
    swap_axes=True,
    dot_min=0,
    figsize=(5, 14),
    return_fig=True
)

# Render the figure internally
dp.make_figure()

scatter_artist = dp.ax_dict['mainplot_ax'].collections[0]

original_sizes = scatter_artist.get_sizes()

# Apply size transformation for visualization
min_visible_size = 20
new_sizes = original_sizes.copy()

new_sizes[new_sizes > 0] += min_visible_size 

scatter_artist.set_sizes(new_sizes)

# Remove the dot borders
scatter_artist.set_linewidths(0)

# save
dp.fig.savefig("CS_Genes_Dotplot_Resized.pdf", bbox_inches='tight')

plt.show()

## CS scoring

### Compute CS scores per cell
---
Fig. S1H

In [14]:
sc.tl.score_genes(
    adata_main, 
    gene_list=cs_genes, 
    score_name='CS_score', 
    use_raw=False,
    ctrl_size=len(cs_genes)
)

computing score 'CS_score'
    finished (0:01:57)


In [63]:
# Calculate the 0.01 and 0.99 quantiles of CS_score
q01 = adata_main.obs['CS_score'].quantile(0.01)
q99 = adata_main.obs['CS_score'].quantile(0.99)
print(f"0.01 quantile: {q01}, 0.99 quantile: {q99}")

0.01 quantile: -0.0911765606469842, 0.99 quantile: 0.5273767303274279


In [ ]:
# A umap for visualizing CS scores overall (Fig. S1H)
gray_red = LinearSegmentedColormap.from_list('gray_red', ['lightgrey', "#D71B1B"])

fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
sc.pl.umap(
    adata_main,
    color='CS_score',
    cmap=gray_red,
    size=0.5,
    vmin = 0,
    vmax = q99,
    show=False,
    frameon=False,
    ax=ax,
    legend_loc='right margin'
)

plt.tight_layout()
plt.savefig('CS_score_major.pdf', dpi=300, bbox_inches='tight')

### CS score correlation
---
Fig. 2D

In [5]:
# Subset three adata objects for each disease group
adata_car_t_correlation = adata_main[adata_main.obs['disease'] == 'CAR-T_CRS'].copy()
adata_covid19_correlation = adata_main[adata_main.obs['disease'] == 'COVID19'].copy()
adata_sle_correlation = adata_main[adata_main.obs['disease'] == 'SLE'].copy()

# Remove No_CRS samples from the CAR-T correlation object
adata_car_t_correlation = adata_car_t_correlation[
    adata_car_t_correlation.obs['severity'] != 'No_CRS'
].copy()

# Remove missing-severity samples from the SLE correlation object
adata_sle_correlation = adata_sle_correlation[
    adata_sle_correlation.obs['severity'].notna()
].copy()

In [ ]:
# Monocyte (all) vs non-myeloid CS score correlation, per disease (Fig. 2D)

# Define diseases to loop through
disease_adatas = {
    'CAR-T_CRS': adata_car_t_correlation,
    'COVID19': adata_covid19_correlation,
    'SLE': adata_sle_correlation
}

# One color per disease
disease_colors = {
    'CAR-T_CRS': '#2076B5',
    'COVID19': '#E37825',
    'SLE': '#349239'  
}

for disease_name, ad in disease_adatas.items():
    obs = ad.obs.copy()

    # Sample-level mean CS score for all non-myeloid cells
    non_myeloid_means = (
        obs[obs['leiden1_anno'] != 'Myeloid']
        .groupby('sample_id', observed=True)['CS_score']
        .mean()
        .rename('non_myeloid_CS_mean')
    )

    # Sample-level mean CS score for all monocytes
    monocyte_means = (
        obs[obs['subcluster_anno'].astype(str).str.startswith('Mono_', na=False)]
        .groupby('sample_id', observed=True)['CS_score']
        .mean()
        .rename('monocyte_CS_mean')
    )

    # Merge and keep valid samples
    df_plot = pd.concat([non_myeloid_means, monocyte_means], axis=1).dropna()

    if len(df_plot) < 3:
        print(f"Skipping {disease_name}: insufficient data points ({len(df_plot)})")
        continue

    # Min-max scale both axes to 0-1 (for visualizations)
    x_raw = df_plot['monocyte_CS_mean']
    y_raw = df_plot['non_myeloid_CS_mean']

    x_range = x_raw.max() - x_raw.min()
    y_range = y_raw.max() - y_raw.min()

    df_plot['x_scaled'] = (x_raw - x_raw.min()) / x_range if x_range > 0 else 0.5
    df_plot['y_scaled'] = (y_raw - y_raw.min()) / y_range if y_range > 0 else 0.5

    # Correlation
    r_pearson, p_pearson = pearsonr(df_plot['x_scaled'], df_plot['y_scaled'])
    r_spearman, p_spearman = spearmanr(df_plot['x_scaled'], df_plot['y_scaled'])

    # Plot
    plt.figure(figsize=(5, 5), dpi=300)
    sns.regplot(
        data=df_plot,
        x='x_scaled',
        y='y_scaled',
        scatter_kws={'s': 50, 'alpha': 0.8, 'color': disease_colors[disease_name]},
        line_kws={'color': disease_colors[disease_name]}
    )

    plt.title(f'CS Score Correlation ({disease_name})')
    plt.xlabel('Mean CS Score (All Monocytes, scaled 0-1)')
    plt.ylabel('Mean CS Score (Non-Myeloid, scaled 0-1)')
    plt.xlim(0, 1)
    plt.ylim(0, 1)

    stats_text = f'Pearson r: {r_pearson:.2f}\np: {p_pearson:.2e}\nSpearman rho: {r_spearman:.2f}'
    plt.text(
        0.05, 0.95, stats_text,
        transform=plt.gca().transAxes,
        va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.6),
        fontsize=9
    )

    plt.tight_layout()
    plt.savefig(f"CS_Correlation_{disease_name}_Monocyte_vs_NonMyeloid.pdf", bbox_inches='tight')
    plt.show()
    plt.close()

print("Sample counts per disease:")
for disease_name, ad in disease_adatas.items():
    print(f"{disease_name}: {ad.obs['sample_id'].nunique()} samples")

### Ridge plot of CS scores
---
Fig. 2A

In [15]:
# CS-related groups
target_groups = ['CAR-T_CRS_pro', 'COVID19_pro', 'SLE']

# New annotation order
anno_order = ['Myeloid', 'Lymphoid']

# Extract relevant data
df = adata_main.obs[['stage', 'leiden1_anno', 'CS_score']].copy()

# Remove missing/"nan" stage entries
df = df[df['stage'].notna() & (df['stage'].astype(str) != 'nan')]

# Filter for groups of interest
df = df[df['stage'].isin(target_groups)]

# Create new annotation column: Myeloid vs Lymphoid
df['anno'] = np.where(df['leiden1_anno'] == 'Myeloid', 'Myeloid', 'Lymphoid')

# Convert group to category with specified order
df['stage'] = pd.Categorical(
    df['stage'],
    categories=target_groups,
    ordered=True
)

# Convert anno to category with specified order
df['anno'] = pd.Categorical(
    df['anno'],
    categories=anno_order,
    ordered=True
)

# Combine group and anno for plotting
df = df.sort_values(['stage', 'anno'])
df['stage_anno'] = df['stage'].astype(str) + '_' + df['anno'].astype(str)

In [ ]:
score_col = 'CS_score'

ridge_order = [
    'CAR-T_CRS_pro_Myeloid', 'CAR-T_CRS_pro_Lymphoid',
    'COVID19_pro_Myeloid', 'COVID19_pro_Lymphoid',
    'SLE_Myeloid', 'SLE_Lymphoid'
]

plot_order = [x for x in ridge_order if x in df['stage_anno'].unique()]
df_plot = df[df['stage_anno'].isin(plot_order)].copy()

group_means_cs = df_plot.groupby('stage_anno')[score_col].median().reindex(plot_order)
norm = colors.Normalize(vmin=group_means_cs.min(), vmax=group_means_cs.max())
cmap = cm.get_cmap('RdBu_r')
palette_cs = {grp: cmap(norm(val)) for grp, val in group_means_cs.items()}

sns.set_theme(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})
g = sns.FacetGrid(
    df_plot,
    row="stage_anno",
    hue="stage_anno",
    row_order=plot_order,
    aspect=10,
    height=0.5,
    palette=palette_cs
)

# filled ridge
g.map(
    sns.kdeplot,
    score_col,
    clip_on=False,
    fill=True,
    alpha=0.9,
    linewidth=0,
    bw_adjust=1.5,
    cut=0
)

# white border line
g.map(
    sns.kdeplot,
    score_col,
    clip_on=False,
    fill=False,
    color="white",
    linewidth=1.1,
    bw_adjust=1.5,
    cut=0
)

g.map(plt.axhline, y=0, lw=0.8, clip_on=False, color='black')

g.figure.subplots_adjust(hspace=-0.5)
g.set_titles("")
g.set(yticks=[], ylabel="", xlabel=None)
g.despine(bottom=True, left=True)

plt.xlim(df_plot[score_col].min(), df_plot[score_col].max())
plt.savefig('RidgePlot_CS_score_by_group_celltype.pdf', bbox_inches='tight', dpi=300)
plt.show()

### Boxplot of CS scores
---
Fig. 1D metadata, Table S2b

In [20]:
# Define the custom sorting key based on the updated hierarchy
def group_sort_key(group_name):
    s = str(group_name)
    
    # Disease Order: CAR-T_CRS, COVID19, SLE, HC
    if 'CAR-T' in s: d_rank = 0
    elif 'COVID' in s: d_rank = 1
    elif 'SLE' in s: d_rank = 2
    elif 'HC' in s or 'Control' in s: d_rank = 3
    else: d_rank = 99
    
    # Severity Order: No_CRS, Moderate, Severe
    if 'No_CRS' in s: s_rank = 0
    elif 'Moderate' in s: s_rank = 1
    elif 'Severe' in s: s_rank = 2
    else: s_rank = 3
    
    # Stage Order: before, pro, con
    if 'before' in s: t_rank = 0
    elif 'pro' in s: t_rank = 1
    elif 'con' in s: t_rank = 2
    else: t_rank = 3 
    
    return (d_rank, s_rank, t_rank)

# Extract unique groups and sort them
unique_groups = [str(g) for g in adata_main.obs['group'].unique() if str(g) != 'nan']
sorted_groups = sorted(unique_groups, key=group_sort_key)


In [21]:
# Per-sample mean CS score by group
score_col = 'CS_score'

df_sample = adata_main.obs[['sample_id', 'group', score_col]].copy()
df_sample['group'] = df_sample['group'].astype(str)
df_sample = df_sample[df_sample['group'] != 'nan']

# Mean CS score per sample within each group
sample_means = (
    df_sample
    .groupby(['group', 'sample_id'], observed=True)[score_col]
    .mean()
    .reset_index(name='sample_mean_CS')
)

# Keep your existing group order if available
group_order = [g for g in sorted_groups if g in sample_means['group'].unique()]

# Build output: each column is a group, values are per-sample mean CS scores
df_sample_mean_wide = pd.DataFrame({
    g: sample_means.loc[sample_means['group'] == g, 'sample_mean_CS'].reset_index(drop=True)
    for g in group_order
})

# Save (metadata for Fig. 1D)
df_sample_mean_wide.to_csv('CS_score_per_sample_mean_by_group.csv', index=False)

# Preview
df_sample_mean_wide.head()

,CAR-T_CRS_before_No_CRS,CAR-T_CRS_pro_No_CRS,CAR-T_CRS_con_No_CRS,CAR-T_CRS_before_Moderate,CAR-T_CRS_pro_Moderate,CAR-T_CRS_con_Moderate,CAR-T_CRS_before_Severe,CAR-T_CRS_pro_Severe,CAR-T_CRS_con_Severe,COVID19_pro_Moderate,COVID19_con_Moderate,COVID19_pro_Severe,COVID19_con_Severe,SLE_Moderate,SLE_Severe,SLE_nan,HC_HC
0,0.107903,0.073323,0.110045,0.225661,0.064114,0.101183,0.161538,0.099625,0.113718,0.046799,0.052691,0.117550,0.138359,0.165229,0.222405,0.197496,0.124608
1,0.039474,0.121539,0.162561,0.156012,0.047402,0.118532,0.184600,0.086429,0.151691,0.049947,0.031678,0.164865,0.177754,0.199632,0.289409,0.117594,0.136276
2,0.111567,0.298205,0.204638,0.068152,0.197059,0.215957,0.056698,0.137201,0.099700,0.040624,0.047556,0.185556,0.136179,0.164597,0.182184,NaN,0.069259
3,0.177255,0.131177,0.193750,0.089716,0.167485,0.176588,0.174692,0.315105,0.103874,0.056784,0.025021,0.107009,0.149903,0.218998,0.174626,NaN,0.081185
4,0.121505,0.273685,0.241220,0.132726,0.159198,0.152898,0.125046,0.174570,0.174729,0.053171,0.157272,0.102751,0.159088,0.198276,0.297440,NaN,0.095453


In [22]:
# CS score summaries across diseases, stages, and severities for all samples (sample-level)
# Define columns (include sample_id)
cols = ['disease', 'stage', 'severity', 'sample_id', 'CS_score']
df = adata_main.obs[cols].copy()

# Drop missing values
df = df.dropna(subset=['disease', 'stage', 'severity', 'sample_id', 'CS_score'])

# compute sample-level CS score (mean per sample)
sample_level = (
    df.groupby(['disease', 'stage', 'severity', 'sample_id'], observed=True)['CS_score']
    .mean()
    .reset_index(name='Sample_CS_score')
)

# Group by condition and compute summary stats on sample-level scores
grouped = sample_level.groupby(['disease', 'stage', 'severity'], observed=True)['Sample_CS_score']

summary = grouped.agg(
    Sample_count='count',  # number of samples
    Mean_CS_score='mean',
    Median_CS_score='median',
    SD_CS_score='std',
    SEM_CS_score='sem',
    Q1_CS_score=lambda x: x.quantile(0.25),
    Q3_CS_score=lambda x: x.quantile(0.75),
    Min_CS_score='min',
    Max_CS_score='max'
).reset_index()

# Rename columns
summary = summary.rename(columns={
    'disease': 'Disease',
    'stage': 'Stage',
    'severity': 'Severity'
})

# remove 0 sample count rows
summary = summary[summary['Sample_count'] > 0]

# Save to Excel (Table S2b)
output_file = "CS_score_summary_by_disease_stage_severity.xlsx"
summary.to_excel(output_file, index=False)

print(f"Saved: {output_file}")

Saved: CS_score_summary_by_disease_stage_severity.xlsx


# 8. CellChat input preparation
---
**Tool:** CellChat
**Aim:** prepare and export expression matrices, gene names, and metadata for the standalone R workflows in `scripts/cellchat/`.

## Data preparation for CAR-T_CRS CellChat
1. Subset the data for CellChat analysis
2. Downsample each group to approximately 50,000 cells while retaining rare annotations

In [ ]:
# Subset to CAR-T_CRS for CellChat input preparation
subset_mask = adata_main.obs['disease'] == 'CAR-T_CRS'
adata_car_t_cellchat = adata_main[subset_mask].copy()

# remove FCb samples
FCb_samples = [
        "CART_S001",
        "CART_S008",
        "CART_S015",
        "CART_S022",
        "CART_S029",
        "CART_S036",
        "CART_S043",
        "CART_S049",
        "CART_S056",
        "CART_S063",
        "CART_S070",
        "CART_S077",
        "CART_S084",
        "CART_S090",
        "CART_S097",
        "CART_S104"]

# Exclude FCb samples
adata_car_t_cellchat = adata_car_t_cellchat[
    ~adata_car_t_cellchat.obs['sample_id'].isin(FCb_samples)
].copy()

# samples that do not have their pair
removed_samples = [
    'CART_S087',
    'CART_S094',
    'CART_S047'
]
adata_car_t_cellchat = adata_car_t_cellchat[
    ~adata_car_t_cellchat.obs['sample_id'].isin(removed_samples)
].copy()

# remove 'No_CRS' samples
adata_car_t_cellchat = adata_car_t_cellchat[
    adata_car_t_cellchat.obs['severity'] != 'No_CRS'
].copy()

In [ ]:
# target monocyte subclusters
target_myeloid = [
    'Mono_CD14_IL1B',
    'Mono_CD14_S100A8',
    'Mono_CD16_LST1'
]

# keep only:
# 1) target monocyte subclusters
# 2) CAR-T cells, excluding endogenous T cells
is_target_mono = adata_car_t_cellchat.obs['subcluster_anno'].isin(target_myeloid)

is_car_t = (
    adata_car_t_cellchat.obs['CAR-T_anno'].astype(str).eq('CAR-T')
)

adata_car_t_cellchat = adata_car_t_cellchat[is_target_mono | is_car_t].copy()

# CellChat annotation:
# monocytes keep monocyte subcluster labels;
# CAR-T cells keep their T-cell subcluster labels
adata_car_t_cellchat.obs['cellchat_anno'] = (
    adata_car_t_cellchat.obs['subcluster_anno'].astype(str)
)

print("Final selected CAR-T_CRS subset:")
print(f"{adata_car_t_cellchat.n_obs} cells")
print(adata_car_t_cellchat.obs['cellchat_anno'].value_counts())

In [ ]:
# Define groups to export
car_t_cellchat_groups = [
    'CAR-T_CRS_before',
    'CAR-T_CRS_pro',
    'CAR-T_CRS_con'
]

def validate_and_export_cellchat_input(sub, save_dir, group_name):
    if sub.n_obs == 0:
        raise ValueError(f"No cells available for CellChat group: {group_name}")
    if not sub.obs_names.is_unique:
        raise ValueError(f"Non-unique cell names in CellChat group: {group_name}")
    if not sub.var_names.is_unique:
        raise ValueError(f"Non-unique gene names in CellChat group: {group_name}")
    if 'cellchat_anno' not in sub.obs.columns or sub.obs['cellchat_anno'].isna().any():
        raise ValueError(f"Invalid cellchat_anno values in group: {group_name}")

    expression_matrix = sub.X.T
    expected_shape = (sub.n_vars, sub.n_obs)
    if expression_matrix.shape != expected_shape:
        raise ValueError(
            f"CellChat matrix shape mismatch for {group_name}: "
            f"{expression_matrix.shape} != {expected_shape}"
        )

    os.makedirs(save_dir, exist_ok=True)
    scipy.io.mmwrite(os.path.join(save_dir, 'expression_matrix.mtx'), expression_matrix)
    sub.obs.to_csv(os.path.join(save_dir, 'meta.csv'))
    pd.DataFrame(sub.var_names).to_csv(
        os.path.join(save_dir, 'genes.csv'), header=False, index=False
    )

    return (
        sub.obs.groupby('cellchat_anno', observed=True).size()
        .rename('n_cells').reset_index()
        .assign(group=group_name, n_genes=sub.n_vars)
    )

cellchat_export_summaries = []
car_t_rng = np.random.default_rng(RANDOM_SEED)

for g_name in car_t_cellchat_groups:
    print(f"\nProcessing {g_name}...")

    sub = adata_car_t_cellchat[
        adata_car_t_cellchat.obs['stage'].astype(str).eq(g_name)
    ].copy()

    print(f" - Selected {sub.n_obs} cells")
    print(sub.obs['cellchat_anno'].value_counts())

    if sub.n_obs == 0:
        print(" - Warning: no cells found, skipping export.")
        continue

    # Keep genes expressed in at least 0.1% of cells in this stage subset.
    sc.pp.filter_genes(sub, min_cells=int(0.001 * sub.n_obs))

    # Downsample to approximately 50,000 cells while retaining rare annotations.
    current_n = sub.n_obs
    if current_n > 50000:
        print(f" - Downsampling {g_name} from {current_n} to approximately 50000 cells")
        fraction = 50000 / current_n
        sampled_indices = []
        for label, cell_group in sub.obs.groupby('cellchat_anno', observed=True):
            n_cells_in_label = len(cell_group)
            n_sample = int(n_cells_in_label * fraction)
            min_cells = 100
            if n_sample < min_cells:
                n_sample = min(n_cells_in_label, min_cells)

            chosen = car_t_rng.choice(
                cell_group.index.to_numpy(), size=n_sample, replace=False
            )
            sampled_indices.extend(chosen)

        sub = sub[sampled_indices, :].copy()
        print(f" - Retained {sub.n_obs} cells after downsampling")
    else:
        print(f" - Cell count ({current_n}) is below target, keeping all cells.")

    save_dir = os.path.join(cellchat_car_t_input_dir, g_name)
    cellchat_export_summaries.append(
        validate_and_export_cellchat_input(sub, save_dir, g_name)
    )

    print(f" - Exported to {save_dir}")

print("\nCAR-T CellChat data export completed.")
display(pd.concat(cellchat_export_summaries, ignore_index=True))

## Data preparation for COVID-19 CellChat
1. Subset the data for cellchat analysis
2. 50k cells per group

In [ ]:
# Subset to COVID-19 progression samples
adata_covid19_cellchat = adata_main[
    adata_main.obs['group'].isin(['COVID19_pro_Severe', 'COVID19_pro_Moderate'])
].copy()

# Subset to only PBMC as sample_type
adata_covid19_cellchat = adata_covid19_cellchat[
    adata_covid19_cellchat.obs['sample_type'] == 'PBMC'
].copy()

In [ ]:
# Create an empty annotation column
adata_covid19_cellchat.obs['cellchat_anno'] = None

# Rename T cells annotations to CD4_T and CD8_T
is_t = adata_covid19_cellchat.obs['leiden1_anno'] == 'T'
is_cd4 = adata_covid19_cellchat.obs['subcluster_anno'].str.contains('CD4', na=False)
is_cd8 = adata_covid19_cellchat.obs['subcluster_anno'].str.contains('CD8', na=False)

adata_covid19_cellchat.obs.loc[is_t & is_cd4, 'cellchat_anno'] = 'CD4_T'
adata_covid19_cellchat.obs.loc[is_t & is_cd8, 'cellchat_anno'] = 'CD8_T'

# For Myeloid cells, assign specific subclusters
mono_of_interest = ['Mono_CD14_IL1B', 'Mono_CD14_S100A8', 'cDC_CD1C', 'Mono_CD16_LST1']
is_target_mono = adata_covid19_cellchat.obs['subcluster_anno'].isin(mono_of_interest)
adata_covid19_cellchat.obs.loc[is_target_mono, 'cellchat_anno'] = (
    adata_covid19_cellchat.obs['subcluster_anno']
)

# For other valid lineages, keep their generic annotations
valid_other = ['NK', 'pDC', 'B', 'Plasma']
is_valid_other = adata_covid19_cellchat.obs['leiden1_anno'].isin(valid_other)
adata_covid19_cellchat.obs.loc[is_valid_other, 'cellchat_anno'] = (
    adata_covid19_cellchat.obs['leiden1_anno']
)

# Keep only the specified COVID-19 CellChat populations
adata_covid19_cellchat = adata_covid19_cellchat[
    adata_covid19_cellchat.obs['cellchat_anno'].notna()
].copy()

In [ ]:
# Prepare input for cellchat (groups to analyze)

covid19_cellchat_groups = ['COVID19_pro_Severe', 'COVID19_pro_Moderate']

covid19_rng = np.random.default_rng(RANDOM_SEED)

for g_name in covid19_cellchat_groups:
    # subset to the specific group
    sub = adata_covid19_cellchat[
        adata_covid19_cellchat.obs['group'] == g_name
    ].copy()
    if sub.n_obs == 0:
        print(f" - Warning: no cells found for {g_name}, skipping export.")
        continue
    # filter genes again, keeping genes expressed in at least 0.1% of cells in this subset
    sc.pp.filter_genes(sub, min_cells=int(0.001 * sub.n_obs))
    # downsampling to 50k cells
    current_n = sub.n_obs
    if current_n > 50000:
        print(f" - Downsampling {g_name} from {current_n} to 50000 cells")

        # fraction needed
        fra = 50000 / current_n
        sampled_indices = []

        # group by cellchat_anno and ensure rare cell types are represented
        for label, cell_group in sub.obs.groupby('cellchat_anno', observed=True):
            n_cells_in_label = len(cell_group)
            n_sample = int(n_cells_in_label * fra)

            # safety bound (at least 100 cells per cell type, or all if less than 100)
            min_cells = 100
            if n_sample < min_cells:
                n_sample = min(n_cells_in_label, min_cells)

            # apply sampling
            chosen = covid19_rng.choice(
                cell_group.index.to_numpy(), size=n_sample, replace=False
            )
            sampled_indices.extend(chosen)

        # create the downsampled subset
        sub = sub[sampled_indices, :].copy()

    else:
        print(f' - cell count ({current_n}) is below target, keeping all.')


    # Export data for cellchat
    # create a separate input folder for this group
    save_dir = os.path.join(cellchat_covid19_input_dir, g_name)
    cellchat_export_summaries.append(
        validate_and_export_cellchat_input(sub, save_dir, g_name)
    )
    
    print(f" - Exported data for group: {g_name} to {save_dir}")

print("COVID-19 CellChat data export completed.")
display(pd.concat(cellchat_export_summaries, ignore_index=True))

## Data preparation for SLE CellChat
1. Subset the data for cellchat analysis
2. 50k cells per group

In [ ]:
# Subset to SLE
adata_sle_cellchat = adata_main[
    adata_main.obs['group'].isin(['SLE_Severe', 'SLE_Moderate'])
].copy()

# Subset to only PBMC as sample_type
adata_sle_cellchat = adata_sle_cellchat[
    adata_sle_cellchat.obs['sample_type'] == 'PBMC'
].copy()

In [ ]:
# Create an empty annotation column
adata_sle_cellchat.obs['cellchat_anno'] = None

# Rename T cells annotations to CD4_T and CD8_T
is_t = adata_sle_cellchat.obs['leiden1_anno'] == 'T'
is_cd4 = adata_sle_cellchat.obs['subcluster_anno'].str.contains('CD4', na=False)
is_cd8 = adata_sle_cellchat.obs['subcluster_anno'].str.contains('CD8', na=False)

adata_sle_cellchat.obs.loc[is_t & is_cd4, 'cellchat_anno'] = 'CD4_T'
adata_sle_cellchat.obs.loc[is_t & is_cd8, 'cellchat_anno'] = 'CD8_T'

# For Myeloid cells, assign specific subclusters
mono_of_interest = ['Mono_CD14_IL1B', 'Mono_CD14_S100A8', 'cDC_CD1C', 'Mono_CD16_LST1']
is_target_mono = adata_sle_cellchat.obs['subcluster_anno'].isin(mono_of_interest)
adata_sle_cellchat.obs.loc[is_target_mono, 'cellchat_anno'] = (
    adata_sle_cellchat.obs['subcluster_anno']
)

# For B cells, assign specific subcluster names
b_of_interest = ['Bn_CD19_TCL1A', 'Bm_CD19_CD27', 'Plasma_TNFRSF17']
is_target_b = adata_sle_cellchat.obs['subcluster_anno'].isin(b_of_interest)
adata_sle_cellchat.obs.loc[is_target_b, 'cellchat_anno'] = (
    adata_sle_cellchat.obs['subcluster_anno']
)
# For other valid lineages, keep their generic annotations
valid_other = ['NK', 'pDC']
is_valid_other = adata_sle_cellchat.obs['leiden1_anno'].isin(valid_other)
adata_sle_cellchat.obs.loc[is_valid_other, 'cellchat_anno'] = (
    adata_sle_cellchat.obs['leiden1_anno']
)

# Keep only the specified SLE CellChat populations
adata_sle_cellchat = adata_sle_cellchat[
    adata_sle_cellchat.obs['cellchat_anno'].notna()
].copy()

In [ ]:
# Prepare input for cellchat (groups to analyze)

sle_cellchat_groups = ['SLE_Severe', 'SLE_Moderate']

sle_rng = np.random.default_rng(RANDOM_SEED)

for g_name in sle_cellchat_groups:
    # subset to the specific group
    sub = adata_sle_cellchat[adata_sle_cellchat.obs['group'] == g_name].copy()
    if sub.n_obs == 0:
        print(f" - Warning: no cells found for {g_name}, skipping export.")
        continue
    # filter genes again, keeping genes expressed in at least 0.1% of cells in this subset
    sc.pp.filter_genes(sub, min_cells=int(0.001 * sub.n_obs))
    # downsampling to 50k cells
    current_n = sub.n_obs
    if current_n > 50000:
        print(f" - Downsampling {g_name} from {current_n} to 50000 cells")

        # fraction needed
        fra = 50000 / current_n
        sampled_indices = []

        # group by cellchat_anno and ensure rare cell types are represented
        for label, cell_group in sub.obs.groupby('cellchat_anno', observed=True):
            n_cells_in_label = len(cell_group)
            n_sample = int(n_cells_in_label * fra)

            # safety bound (at least 100 cells per cell type, or all if less than 100)
            min_cells = 100
            if n_sample < min_cells:
                n_sample = min(n_cells_in_label, min_cells)

            # apply sampling
            chosen = sle_rng.choice(
                cell_group.index.to_numpy(), size=n_sample, replace=False
            )
            sampled_indices.extend(chosen)

        # create the downsampled subset
        sub = sub[sampled_indices, :].copy()

    else:
        print(f' - cell count ({current_n}) is below target, keeping all.')


    # Export data for cellchat
    # create a separate input folder for this group
    save_dir = os.path.join(cellchat_sle_input_dir, g_name)
    cellchat_export_summaries.append(
        validate_and_export_cellchat_input(sub, save_dir, g_name)
    )
    
    print(f" - Exported data for group: {g_name} to {save_dir}")

print("SLE CellChat data export completed.")
cellchat_export_summary = pd.concat(cellchat_export_summaries, ignore_index=True)
display(cellchat_export_summary)
cellchat_export_summary.to_csv('CellChat_input_export_summary.csv', index=False)

# 9. SLE pDC analysis

In [26]:
adata_pdc_deg = adata_main[
    adata_main.obs['subcluster_anno'] == 'pDC_LILRA4'
].copy()

# Subset to SLE and HC
adata_pdc_deg = adata_pdc_deg[
    adata_pdc_deg.obs['group'].isin(['SLE_Moderate', 'SLE_Severe'])
].copy()
print(adata_pdc_deg.obs['group'].value_counts())

group
SLE_Moderate    281
SLE_Severe      164
Name: count, dtype: int64


In [27]:
# Perform cell-level DE analysis between SLE_Severe and SLE_Moderate pDCs

# Ensure 'group' is categorical with the correct order
adata_pdc_deg.obs['group'] = pd.Categorical(
    adata_pdc_deg.obs['group'],
    categories=['SLE_Moderate', 'SLE_Severe'],
    ordered=True
)
# Run DE analysis using Scanpy's rank_genes_groups
sc.tl.rank_genes_groups(
    adata_pdc_deg,
    groupby='group',
    groups=['SLE_Severe'],
    reference='SLE_Moderate',
    method='wilcoxon',
    use_raw=False
)
# Extract results
pdc_deg_results = sc.get.rank_genes_groups_df(
    adata_pdc_deg, group='SLE_Severe'
)
# Save to CSV
pdc_deg_results.to_csv('pDC_DE_SLE_Severe_vs_Moderate.csv', index=False)

ranking genes
    finished (0:00:01)


In [28]:
# Subset to pDC + target monocyte subclusters
mono_subclusters = ['Mono_CD14_IL1B', 'Mono_CD14_S100A8', 'Mono_CD16_LST1']

mask_pdc = (
    adata_main.obs['subcluster_anno'].astype(str).str.startswith('pDC', na=False) |
    (adata_main.obs['leiden1_anno'].astype(str) == 'pDC')
)
mask_mono = adata_main.obs['subcluster_anno'].isin(mono_subclusters)

adata_pdc_mono_ifn = adata_main[mask_pdc | mask_mono].copy()

# Subset to groups of interest
pdc_ifn_target_groups = ['SLE_Severe', 'SLE_Moderate', 'HC_HC']
adata_pdc_mono_ifn = adata_pdc_mono_ifn[
    adata_pdc_mono_ifn.obs['group'].isin(pdc_ifn_target_groups)
].copy()

# Remove samples with low pDC counts
min_pdc_cells = 5

is_pdc = (
    adata_pdc_mono_ifn.obs['subcluster_anno'].astype(str).str.startswith('pDC', na=False) |
    (adata_pdc_mono_ifn.obs['leiden1_anno'].astype(str) == 'pDC')
)

pdc_counts = adata_pdc_mono_ifn.obs.loc[is_pdc].groupby('sample_id', observed=True).size()
valid_samples = pdc_counts[pdc_counts >= min_pdc_cells].index

adata_pdc_mono_ifn = adata_pdc_mono_ifn[
    adata_pdc_mono_ifn.obs['sample_id'].isin(valid_samples)
].copy()

# Recompute after filtering for accurate summary
is_pdc_final = (
    adata_pdc_mono_ifn.obs['subcluster_anno'].astype(str).str.startswith('pDC', na=False) |
    (adata_pdc_mono_ifn.obs['leiden1_anno'].astype(str) == 'pDC')
)

print(f"Total cells kept (pDC + monocytes): {adata_pdc_mono_ifn.n_obs}")
print(f"pDC cells kept: {is_pdc_final.sum()}")
print(f"Samples kept: {adata_pdc_mono_ifn.obs['sample_id'].nunique()}")

Total cells kept (pDC + monocytes): 67045
pDC cells kept: 1187
Samples kept: 64


## pDC IFN-I scoring
---
Fig. 5D, Table S6c

In [29]:
# pDC_IFN_activation score
pdc_IFN_genes = ['IRF7', 'RSAD2', 'STAT1', 'STAT2']

# Identify pDC cells in the combined pDC-monocyte analysis object
pdc_mask = adata_pdc_mono_ifn.obs['subcluster_anno'].astype(str).str.startswith('pDC', na=False)

print(f"Total cells in adata_pdc_mono_ifn: {adata_pdc_mono_ifn.n_obs}")
print(f"pDC cells to score: {int(pdc_mask.sum())}")

pdc_IFN_genes = require_complete_gene_set(
    adata_pdc_mono_ifn, pdc_IFN_genes, 'pDC IFN activation gene set'
)
if pdc_mask.sum() == 0:
    raise ValueError('No pDC cells found in adata_pdc_mono_ifn.')

sc.tl.score_genes(
    adata_pdc_mono_ifn,
    gene_list=pdc_IFN_genes,
    score_name='pDC_IFN_score',
    use_raw=False,
    ctrl_size=len(pdc_IFN_genes)
)

# Keep score only for pDC cells; non-pDC remain NaN
adata_pdc_mono_ifn.obs.loc[~pdc_mask, 'pDC_IFN_score'] = np.nan
print(f"Computed pDC_IFN_score for {int(pdc_mask.sum())} pDC cells.")

# Calculate per sample means and save into csv
# Build a long table of pDC IFN-I scores
df_long = (
    adata_pdc_mono_ifn.obs.loc[
        pdc_mask & adata_pdc_mono_ifn.obs['pDC_IFN_score'].notna(),
        ['group', 'sample_id', 'pDC_IFN_score']
    ]
    .copy()
)

# Per-sample mean IFN-I score (within each group)
df_sample_mean = (
    df_long.groupby(['group', 'sample_id'], observed=True)['pDC_IFN_score']
    .mean()
    .reset_index(name='pDC_IFN_score_mean')
)

# Stack sample means by group into columns (unequal lengths padded with NaN)
group_order = [
    g for g in ['SLE_Severe', 'SLE_Moderate', 'HC_HC']
    if g in df_sample_mean['group'].unique()
]
df_group_cols = pd.DataFrame({
    g: df_sample_mean.loc[df_sample_mean['group'] == g, 'pDC_IFN_score_mean'].reset_index(drop=True)
    for g in group_order
})

# Save without row index
df_group_cols.to_csv('pDC_IFN_sample_means_by_group.csv', index=False)

print("Saved: pDC_IFN_sample_means_by_group.csv")
print(df_group_cols.head())

# Generate another csv table with: sample_id, group, pDC_IFN_score_mean
df_sample_mean.to_csv('pDC_IFN_sample_means_long.csv', index=False)

Total cells in adata_pdc_mono_ifn: 67045
pDC cells to score: 1187
computing score 'pDC_IFN_score'
    finished (0:00:01)
Computed pDC_IFN_score for 1187 pDC cells.
Saved: pDC_IFN_sample_means_by_group.csv
   SLE_Severe  SLE_Moderate     HC_HC
0    0.715357      0.487021  0.440868
1    0.773967      0.350996  0.668689
2    0.217424      0.683812  0.303227
3    0.373857      0.745333  0.271306
4    0.673590      0.674157  0.639624


## Dotplot of pDC scoring genes
---
Fig. S5E

In [ ]:
# Restrict to the pDC subcluster before plotting pDC IFN-I genes
adata_pdc_dotplot = adata_pdc_mono_ifn[
    pdc_mask & adata_pdc_mono_ifn.obs['group'].isin(
        ['SLE_Severe', 'SLE_Moderate', 'HC_HC']
    )
].copy()
pdc_dotplot = sc.pl.dotplot(
    adata_pdc_dotplot,
    var_names=pdc_IFN_genes,
    groupby='group',
    standard_scale='var',
    dot_max=1,
    dot_min=0,
    cmap='OrRd',
    show=False,
    return_fig=True
)
pdc_dotplot.savefig('pDC_IFN_genes_dotplot_SLE.pdf', bbox_inches='tight')

## Monocyte IFN response and pDC IFN activation scores correlation
---
Fig. 5E

In [31]:
# Score monocyte IFN-I response
monocyte_genes = ['ISG15', 'IFITM3', 'IFI44L', 'IFIT1', 'IFI6', 'MX1', 'SIGLEC1']

# Identify monocyte cells in the combined pDC-monocyte analysis object
monocyte_mask = adata_pdc_mono_ifn.obs['subcluster_anno'].astype(str).str.startswith('Mono', na=False)

print(f"Total cells in adata_pdc_mono_ifn: {adata_pdc_mono_ifn.n_obs}")
print(f"Monocyte cells to score: {int(monocyte_mask.sum())}")

monocyte_genes = require_complete_gene_set(
    adata_pdc_mono_ifn, monocyte_genes, 'monocyte IFN-I response gene set'
)
if monocyte_mask.sum() == 0:
    raise ValueError('No monocyte cells found in adata_pdc_mono_ifn.')

sc.tl.score_genes(
    adata_pdc_mono_ifn,
    gene_list=monocyte_genes,
    score_name='IFN_I_response',
    use_raw=False,
    ctrl_size=len(monocyte_genes)
)

# Keep score only for monocyte cells; non-monocyte remain NaN
adata_pdc_mono_ifn.obs.loc[~monocyte_mask, 'IFN_I_response'] = np.nan
print(f"Computed IFN_I_response for {int(monocyte_mask.sum())} monocyte cells.")


Total cells in adata_pdc_mono_ifn: 67045
Monocyte cells to score: 65858
computing score 'IFN_I_response'
    finished (0:00:01)
Computed IFN_I_response for 65858 monocyte cells.


In [ ]:
# Plot sample-level pDC-versus-monocyte IFN-I score correlations

groups = ['SLE_Severe', 'SLE_Moderate', 'HC_HC']

required_score_columns = {'pDC_IFN_score', 'IFN_I_response'}
missing_score_columns = required_score_columns - set(adata_pdc_mono_ifn.obs.columns)
if missing_score_columns:
    raise ValueError(
        f"Run the preceding scoring cells first; missing columns: {sorted(missing_score_columns)}"
    )

obs = adata_pdc_mono_ifn.obs.copy()

# Consistent cell masks
pdc_mask = (
    obs['subcluster_anno'].astype(str).str.startswith('pDC', na=False) |
    (obs['leiden1_anno'].astype(str) == 'pDC')
)
mono_mask = obs['subcluster_anno'].astype(str).str.startswith('Mono', na=False)

print(f"pDC cells: {int(pdc_mask.sum())}, Monocytes: {int(mono_mask.sum())}")

# Scores were restricted to their intended cell populations in the preceding cells.
# Build sample-level means from the resulting non-missing values.

pdc_sample_mean = (
    obs.loc[pdc_mask, ['group', 'sample_id', 'pDC_IFN_score']]
    .dropna()
    .groupby(['group', 'sample_id'], observed=True)['pDC_IFN_score']
    .mean()
    .rename('pdc_IFN_I_mean')
)

mono_sample_mean = (
    obs.loc[mono_mask, ['group', 'sample_id', 'IFN_I_response']]
    .dropna()
    .groupby(['group', 'sample_id'], observed=True)['IFN_I_response']
    .mean()
    .rename('mono_IFN_I_response_mean')
)

corr_df = pd.concat([pdc_sample_mean, mono_sample_mean], axis=1).dropna().reset_index()
print(f"Rows in corr_df: {len(corr_df)}")

if corr_df.empty:
    display(
        obs.groupby('group', observed=True)[['pDC_IFN_score', 'IFN_I_response']]
        .apply(lambda x: x.notna().sum())
    )
    raise ValueError("corr_df is still empty after rescoring.")

plot_groups = [g for g in groups if g in corr_df['group'].unique()]

fig, axes = plt.subplots(2, 2, figsize=(6, 6), dpi=300)
axes = axes.flatten()

for i, g in enumerate(plot_groups[:4]):
    ax = axes[i]
    df_g = corr_df[corr_df['group'] == g].copy()

    if len(df_g) < 3:
        ax.text(0.5, 0.5, f"{g}\nInsufficient samples (n={len(df_g)})",
                ha='center', va='center', transform=ax.transAxes, fontsize=10)
        ax.set_axis_off()
        continue

    sns.regplot(
        data=df_g,
        x='pdc_IFN_I_mean',
        y='mono_IFN_I_response_mean',
        ax=ax,
        scatter_kws={'s': 20, 'alpha': 0.8},
        line_kws={'color': 'red', 'lw': 1.5}
    )

    r, p = pearsonr(df_g['pdc_IFN_I_mean'], df_g['mono_IFN_I_response_mean'])
    ax.set_title(f"{g} (n={len(df_g)})", fontsize=10)
    ax.set_xlabel("pDC IFN-I score (sample mean)")
    ax.set_ylabel("Monocyte IFN-I response (sample mean)")
    ax.text(
        0.05, 0.95, f"Pearson r = {r:.2f}\np = {p:.2e}",
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.6), fontsize=9
    )
    ax.grid(False)

for j in range(len(plot_groups[:4]), 4):
    axes[j].set_axis_off()

plt.tight_layout()
plt.savefig("pDC_vs_Monocyte_IFN_correlation_by_group.pdf", bbox_inches='tight', dpi=300)
plt.show()

# 10. IL6 expr by B cells
---
Fig. S5J

In [ ]:
# Keep groups of interest
adata_il6 = adata_main[adata_main.obs['group'].isin(['SLE_Severe', 'SLE_Moderate', 'COVID19_pro_Severe', 'COVID19_pro_Moderate', 'HC_HC'])].copy()

# Subset to only B cells and Plasma cells
adata_il6 = adata_il6[adata_il6.obs['leiden1_anno'].isin(['B', 'Plasma'])].copy()

# Remove SLE_nan
adata_il6 = adata_il6[adata_il6.obs['severity'] != 'nan'].copy()

# Build plotting dataframe
if 'IL6' not in adata_il6.var_names:
    raise ValueError("IL6 not found in adata_sub.var_names")

il6_df = adata_il6.to_df()[['IL6']]
plot_df = adata_il6.obs[['sample_id', 'group', 'subcluster_anno']].join(il6_df)

# Per-sample mean IL6 within each (group, subcluster), then average across samples in each group
sample_level = (
    plot_df
    .groupby(['sample_id', 'group', 'subcluster_anno'], observed=True)['IL6']
    .mean()
    .reset_index()
)

heatmap_df = (
    sample_level
    .groupby(['subcluster_anno', 'group'], observed=True)['IL6']
    .mean()
    .unstack('group')
)

# Column order
group_order = ['SLE_Severe', 'SLE_Moderate', 'COVID19_pro_Severe', 'COVID19_pro_Moderate', 'HC_HC']
group_order = [g for g in group_order if g in heatmap_df.columns]
heatmap_df = heatmap_df.reindex(columns=group_order)

# Optional row order by overall IL6 level
heatmap_df = heatmap_df.loc[heatmap_df.mean(axis=1).sort_values(ascending=False).index]

# Plot heatmap
plt.figure(figsize=(8, 5))
sns.heatmap(
    heatmap_df,
    cmap='RdBu_r',
    annot=True,
    fmt='.2f',
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Mean IL6 expression'}
)
plt.title('IL6 Expression Across Groups by B/Plasma Subcluster')
plt.xlabel('Group')
plt.ylabel('Subcluster (subcluster_anno)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('IL6_subcluster_group_heatmap.pdf', dpi=300, bbox_inches='tight')
plt.show()